In [ ]:
# NB40 V8 - stable baseline + canonical destination resolver + Fabric-safe media dedup
# - Based on the last stable MEDIA_CONTRACT_FIXED notebook, not V4/V5/V6.
# - Preserves the established namespace/execution contract.
# - Removes correlated media aggregation and deduplicates in CTEs.
# - Destination resolver: ward + city; Phú Quốc special parsing for An Thới/Dương Đông.

# Fabric Notebook: NB_40_GOLD_POI_SERVING
# READ : Silver (+ existing Gold destination reference tables)
# WRITE: Gold poi_master, poi_detail, poi_rating, poi_review, poi_enrichment_display

run_id = globals().get("run_id", "DEV_RUN")
STAGE = "_stage_poi_language"

# Only impacted logical POIs are rebuilt.
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW v2_impacted_poi AS
SELECT DISTINCT poi_id,poi_group_id,lang FROM {STAGE} WHERE run_id='{run_id}'
""")

# Optional migration columns
for tbl in ["sgr_visitvn_dev.lh_vv_gold.dbo.poi_master", "sgr_visitvn_dev.lh_vv_gold.dbo.poi_detail"]:
    try:
        spark.sql(f"ALTER TABLE {tbl} ADD COLUMNS (poi_group_id STRING)")
    except Exception:
        pass

# -----------------------------------------------------------------------------
# 7A.2 BUSINESS CONTRACT PREFLIGHT
#
# Business source of truth:
# poi_enrichment -> poi_enrichment_display
#
# NB40 intentionally does NOT infer these canonical fields from legacy columns.
# If they are absent, NB30 / poi_enrichment schema must be aligned first.
# -----------------------------------------------------------------------------
_REQUIRED_7A2_POI_ENRICHMENT_COLS = {
    "enrichment_id",
    "poi_id",
    "poi_name",
    "lang",
    "ref_business_sector",
    "description",
    "time_to_spend_minutes",
    "best_time_type",
    "best_time_from_month",
    "best_time_to_month",
    "cost_vnd",
    "experiences_json",
    "enriched_at",
    "is_current",
    "_created_at",
    "_updated_at"
}

_actual_enrichment_cols = set(spark.table("poi_enrichment").columns)
_missing_7a2_cols = sorted(
    _REQUIRED_7A2_POI_ENRICHMENT_COLS - _actual_enrichment_cols
)

if _missing_7a2_cols:
    raise RuntimeError(
        "NB40_7A2_SOURCE_SCHEMA_MISMATCH: poi_enrichment is missing canonical "
        f"business fields required by 7A.2: {_missing_7a2_cols}. "
        "Required canonical 7A.2 source fields are missing. "
        "Align NB30/poi_enrichment first."
    )

print("NB40 7A.2 source schema preflight = OK")

# -----------------------------------------------------------------------------
# NB40 PHYSICAL-SCHEMA PREFLIGHT
# Fail before any Gold MERGE if a target schema drifts.
# -----------------------------------------------------------------------------
_TARGET_SCHEMAS = {
    "sgr_visitvn_dev.lh_vv_gold.dbo.poi_master": {
        "poi_id","lang","slug","poi_name","business_sector","business_category",
        "subcategory_tags","operating_status","is_active","is_verified",
        "short_description","short_address","district","rating_overall","rating_count",
        "destination_ids","nearby_pois","destination_names","created_at","updated_at",
        "version","source_id","event_id","ref_type","ref_business_sector",
        "ingestion_source","partner_id","is_bookable","poi_group_id"
    },
    "sgr_visitvn_dev.lh_vv_gold.dbo.poi_detail": {
        "poi_id","lang","amenities_json","poi_address_json","poi_contact_json",
        "poi_content_json","poi_media_json","poi_opening_hours_json","poi_policy_json",
        "poi_rating_json","poi_review_json","poi_extra_json","source_id","event_id",
        "created_at","updated_at","ingestion_source","poi_group_id"
    },
    "sgr_visitvn_dev.lh_vv_gold.dbo.poi_rating": {
        "poi_id","overall","total_reviews","score_label","sources_json","breakdown_json",
        "sentiment_avg","source_id","event_id","updated_at","_batch_id"
    },
    "sgr_visitvn_dev.lh_vv_gold.dbo.poi_review": {
        "review_id","poi_id","source","source_review_id","lang","reviewer_name","rating",
        "title","positive","negative","review_text","review_date","review_trip_type",
        "sentiment_score","ranking_text","awards_json","response_text","responded_at",
        "media_urls_json","helpful_votes","published_at","last_projected_at",
        "source_id","event_id","_batch_id"
    },
    "sgr_visitvn_dev.lh_vv_gold.dbo.poi_enrichment_display": {
        "display_id","enrichment_id","poi_id","poi_name","lang","display_category",
        "description","time_to_spend","best_time","cost","experiences",
        "enrichment_job_id","validation_status","display_generated_at","is_current",
        "ingestion_source","created_at","updated_at"
    }
}

for _tbl, _expected in _TARGET_SCHEMAS.items():
    _actual = set(spark.table(_tbl).columns)
    _missing = sorted(_expected - _actual)
    _unexpected = sorted(_actual - _expected)
    if _missing:
        raise RuntimeError(
            f"NB40_SCHEMA_PREFLIGHT_FAILED table={_tbl} "
            f"missing={_missing} unexpected={_unexpected}"
        )
    print(
        f"NB40 schema preflight OK: {_tbl} "
        f"(columns={len(_actual)}, unexpected={_unexpected})"
    )

# -----------------------------------------------------------------------------
# DESTINATION RESOLVER - canonical source of truth
#
# Normal rule:
#   poi_address.ward = destination_ward_mapping.ward
#   poi_address.city = destination_ward_mapping.province_name
#
# Phu Quoc special rule:
#   - if poi_address.ward = "Phú Quốc"
#   - parse the 3rd comma-separated component from the RIGHT of address_line1
#   - remove leading "Phường" / "Xã"
#   - if parsed ward is "An Thới" or "Dương Đông": lookup ONLY by ward
#   - otherwise fallback to the normal ward + city rule
#
# Notes:
#   - matching is case-insensitive, trim-safe, repeated-space-safe
#   - accents are preserved; this resolver does not perform fuzzy matching
#   - mapping is deterministically deduplicated by dest_id
# -----------------------------------------------------------------------------
spark.sql("""
CREATE OR REPLACE TEMP VIEW v2_destination_resolved AS
WITH impacted_address AS (
    SELECT
        a.poi_id,
        trim(a.ward) AS ward,
        trim(a.city) AS city,
        trim(a.address_line1) AS address_line1,

        lower(trim(regexp_replace(coalesce(a.ward, ''), '\\s+', ' '))) AS ward_norm,
        lower(trim(regexp_replace(coalesce(a.city, ''), '\\s+', ' '))) AS city_norm,

        trim(
            regexp_replace(
                trim(element_at(split(coalesce(a.address_line1, ''), ','), -3)),
                '(?i)^\\s*(phường|xã)\\s+',
                ''
            )
        ) AS phu_quoc_parsed_ward
    FROM poi_address a
    INNER JOIN v2_impacted_poi i
        ON i.poi_id = a.poi_id
),
address_rule AS (
    SELECT
        *,
        lower(
            trim(
                regexp_replace(
                    coalesce(phu_quoc_parsed_ward, ''),
                    '\\s+',
                    ' '
                )
            )
        ) AS phu_quoc_parsed_ward_norm,

        CASE
            WHEN ward_norm = 'phú quốc'
             AND lower(
                    trim(
                        regexp_replace(
                            coalesce(phu_quoc_parsed_ward, ''),
                            '\\s+',
                            ' '
                        )
                    )
                 ) IN ('an thới', 'dương đông')
            THEN true
            ELSE false
        END AS use_phu_quoc_special_rule
    FROM impacted_address
),
mapping_normal AS (
    SELECT ward_norm, province_name_norm, dest_id, dest_name
    FROM (
        SELECT
            lower(trim(regexp_replace(ward, '\\s+', ' '))) AS ward_norm,
            lower(trim(regexp_replace(province_name, '\\s+', ' '))) AS province_name_norm,
            dest_id,
            dest_name,
            ROW_NUMBER() OVER (
                PARTITION BY
                    lower(trim(regexp_replace(ward, '\\s+', ' '))),
                    lower(trim(regexp_replace(province_name, '\\s+', ' ')))
                ORDER BY dest_id
            ) AS rn
        FROM sgr_visitvn_dev.lh_vv_gold.dbo.destination_ward_mapping
        WHERE ward IS NOT NULL
          AND province_name IS NOT NULL
          AND dest_id IS NOT NULL
    ) x
    WHERE rn = 1
),
mapping_special_ward AS (
    SELECT ward_norm, dest_id, dest_name
    FROM (
        SELECT
            lower(trim(regexp_replace(ward, '\\s+', ' '))) AS ward_norm,
            dest_id,
            dest_name,
            ROW_NUMBER() OVER (
                PARTITION BY lower(trim(regexp_replace(ward, '\\s+', ' ')))
                ORDER BY dest_id
            ) AS rn
        FROM sgr_visitvn_dev.lh_vv_gold.dbo.destination_ward_mapping
        WHERE ward IS NOT NULL
          AND dest_id IS NOT NULL
          AND lower(trim(regexp_replace(ward, '\\s+', ' ')))
                IN ('an thới', 'dương đông')
    ) x
    WHERE rn = 1
)
SELECT
    a.poi_id,
    a.ward,
    a.city,
    a.address_line1,
    a.phu_quoc_parsed_ward,
    a.use_phu_quoc_special_rule,

    CASE
        WHEN a.use_phu_quoc_special_rule THEN ms.dest_id
        ELSE mn.dest_id
    END AS dest_id,

    CASE
        WHEN a.use_phu_quoc_special_rule THEN ms.dest_name
        ELSE mn.dest_name
    END AS dest_name,

    CASE
        WHEN a.use_phu_quoc_special_rule THEN 'PHU_QUOC_WARD_ONLY'
        ELSE 'WARD_CITY'
    END AS destination_match_rule

FROM address_rule a

LEFT JOIN mapping_normal mn
    ON a.use_phu_quoc_special_rule = false
   AND a.ward_norm = mn.ward_norm
   AND a.city_norm = mn.province_name_norm

LEFT JOIN mapping_special_ward ms
    ON a.use_phu_quoc_special_rule = true
   AND a.phu_quoc_parsed_ward_norm = ms.ward_norm
""")

# 1. poi_master - impacted-only and language-specific.
# Destination is resolved once by v2_destination_resolved above.
# Normal lookup uses ward + city; Phu Quoc has the documented special parser.
spark.sql("""
CREATE OR REPLACE TEMP VIEW v2_gold_master_src AS
WITH entity AS (
 SELECT e.* FROM poi_entity e JOIN v2_impacted_poi i ON i.poi_id=e.poi_id
 WHERE e._is_current=true AND e.poi_name IS NOT NULL
), content_pick AS (
 SELECT * FROM (
   SELECT c.*,e.lang,
     ROW_NUMBER() OVER(PARTITION BY c.poi_id ORDER BY CASE WHEN c.locale=e.lang THEN 0 WHEN c.locale='vi' THEN 1 WHEN c.locale='en' THEN 2 ELSE 3 END) rn
   FROM poi_content c JOIN entity e ON e.poi_id=c.poi_id
 ) WHERE rn=1
), projected AS (
 SELECT e.poi_id,e.poi_group_id,e.lang,e.slug,e.poi_name,e.business_sector,e.business_category,
   to_json(e.subcategory_tags) subcategory_tags,e.operating_status,e.is_active,e.is_verified,
   c.description_short short_description,COALESCE(a.short_address,a.full_address) short_address,a.district,
   r.overall rating_overall,COALESCE(r.total_reviews,0) rating_count,
   CASE WHEN dm.dest_id IS NOT NULL THEN to_json(array(dm.dest_id)) END destination_ids,
   CASE WHEN dm.dest_name IS NOT NULL THEN to_json(array(dm.dest_name)) END destination_names,
   e.canonical_source ingestion_source,e.partner_id,e.source_id,e.event_id,e._created_at created_at,e._updated_at updated_at,
   CAST(NULL AS INT) version,dm.dest_id _dest_id_for_join
 FROM entity e
 INNER JOIN poi_source_map sm ON sm.poi_id=e.poi_id
 LEFT JOIN poi_address a ON a.poi_id=e.poi_id
 LEFT JOIN poi_rating r ON r.poi_id=e.poi_id
 LEFT JOIN content_pick c ON c.poi_id=e.poi_id
 LEFT JOIN v2_destination_resolved dm ON dm.poi_id=e.poi_id
), nearby AS (
 SELECT p.poi_id,to_json(collect_list(a.poi_id)) nearby_pois
 FROM projected p JOIN anchor_poi_map a ON a.destination_id=p._dest_id_for_join AND a.poi_id<>p.poi_id
 GROUP BY p.poi_id
)
SELECT p.*,n.nearby_pois FROM projected p LEFT JOIN nearby n ON n.poi_id=p.poi_id
""")

spark.sql("""
MERGE INTO sgr_visitvn_dev.lh_vv_gold.dbo.poi_master tgt
USING v2_gold_master_src src ON tgt.poi_id=src.poi_id
WHEN MATCHED THEN UPDATE SET
 tgt.poi_group_id=src.poi_group_id,tgt.lang=src.lang,tgt.slug=src.slug,tgt.poi_name=src.poi_name,
 tgt.business_sector=src.business_sector,tgt.business_category=src.business_category,tgt.subcategory_tags=src.subcategory_tags,
 tgt.operating_status=src.operating_status,tgt.is_active=src.is_active,tgt.is_verified=src.is_verified,
 tgt.short_description=src.short_description,tgt.short_address=src.short_address,tgt.district=src.district,
 tgt.rating_overall=src.rating_overall,tgt.rating_count=src.rating_count,
 tgt.destination_ids=src.destination_ids,tgt.destination_names=src.destination_names,
 tgt.nearby_pois=COALESCE(src.nearby_pois,tgt.nearby_pois),tgt.ingestion_source=src.ingestion_source,
 tgt.partner_id=src.partner_id,tgt.source_id=src.source_id,tgt.event_id=src.event_id,tgt.updated_at=src.updated_at
WHEN NOT MATCHED THEN INSERT(
 poi_id,poi_group_id,lang,slug,poi_name,business_sector,business_category,subcategory_tags,operating_status,is_active,is_verified,
 short_description,short_address,district,rating_overall,rating_count,destination_ids,destination_names,nearby_pois,ingestion_source,
 partner_id,source_id,event_id,created_at,updated_at,version,is_bookable
) VALUES(
 src.poi_id,src.poi_group_id,src.lang,src.slug,src.poi_name,src.business_sector,src.business_category,src.subcategory_tags,
 src.operating_status,src.is_active,src.is_verified,src.short_description,src.short_address,src.district,src.rating_overall,src.rating_count,
 src.destination_ids,src.destination_names,src.nearby_pois,src.ingestion_source,src.partner_id,src.source_id,src.event_id,
 src.created_at,src.updated_at,COALESCE(src.version,0),false
)
""")

# -----------------------------------------------------------------------------
# MEDIA DEDUP CONTRACT - downstream consumer
#
# NB20 is the ONLY authoritative writer of media business identity:
#   media_dedup_key = SHA256(source_name|source_id|media_id)
#
# NB40 must not recompute the key from photo_api_uri/original_url. It only
# consumes the persisted key and performs a defensive POI-scoped dedup before
# building poi_detail.poi_media_json.
#
# Historical rows that predate media_dedup_key fall back to media_id only.
# This fallback preserves historical rows without redefining original_url as a
# business identity.
# -----------------------------------------------------------------------------
_media_cols = set(spark.table("poi_media").columns)
if "media_dedup_key" not in _media_cols:
    raise RuntimeError(
        "NB40_MEDIA_SCHEMA_MISMATCH: poi_media.media_dedup_key is required. "
        "Run the updated NB20 before NB40."
    )

_MEDIA_IDENTITY_SQL = (
    "COALESCE(NULLIF(TRIM(m.media_dedup_key), ''), CONCAT('LEGACY_ID:', m.media_id))"
)

print(
    "NB40 media contract = poi_id + persisted media_dedup_key "
    "(NB20 authoritative; media_id fallback only for historical rows)"
)

# 2. poi_detail - packed API document, impacted only.
# Physical Gold schema:
#   poi_id, lang, amenities_json, poi_address_json, poi_contact_json,
#   poi_content_json, poi_media_json, poi_opening_hours_json, poi_policy_json,
#   poi_rating_json, poi_review_json, poi_extra_json, source_id, event_id,
#   created_at, updated_at, ingestion_source, poi_group_id
#
# Optional Silver components are LEFT JOINed. They must never remove a POI.
# Gold poi_detail JSON contract:
#   object JSON columns -> '{}' when source data is absent
#   poi_review_json     -> '[]' when source data is absent
#   poi_media_json      -> {"cover":null,"gallery":[],"total_count":0}
#
# poi_media_json business contract:
# {
#   "cover":  {"media_id": "...", "blob_url": "..."} | null,
#   "gallery": [{"media_id": "...", "blob_url": "..."}],
#   "total_count": N
# }
# Cover priority: is_blessed=true -> display_order ASC -> media_id ASC.
# Gallery contains all active media, including the selected cover.
# This prevents downstream PostgreSQL NOT NULL violations.
# Amenities parity with the legacy pipeline:
#   poi_amenity_current.resolved_amenities_json -> poi_detail.amenities_json
#   fallback '{}' when the amenity row/payload is missing.
# Content source:
#   Silver poi_content -> poi_detail.poi_content_json
#   fallback '{}' only when Silver content is still absent.
# Policy source:
#   Silver poi_policy.policy_json -> poi_detail.poi_policy_json
#   wrapped as {"policy": ..., "source_updated_at": ...} for legacy parity,
#   fallback '{}' because poi_detail.poi_policy_json is NOT NULL.
# Extra source:
#   Silver poi_price -> poi_detail.poi_extra_json.price
#   fallback '{}' because poi_detail.poi_extra_json is NOT NULL.
_detail_sql = """
CREATE OR REPLACE TEMP VIEW v2_gold_detail_src AS
WITH content_pick AS (
    SELECT *
    FROM (
        SELECT
            c.*,
            e.lang AS entity_lang,
            ROW_NUMBER() OVER (
                PARTITION BY c.poi_id
                ORDER BY
                    CASE
                        WHEN c.locale = e.lang THEN 0
                        WHEN c.locale = 'vi' THEN 1
                        WHEN c.locale = 'en' THEN 2
                        ELSE 3
                    END,
                    c.content_updated_at DESC NULLS LAST,
                    c._updated_at DESC NULLS LAST
            ) AS rn
        FROM poi_content c
        INNER JOIN poi_entity e
            ON e.poi_id = c.poi_id
           AND e._is_current = true
        INNER JOIN v2_impacted_poi i
            ON i.poi_id = c.poi_id
    ) x
    WHERE rn = 1
),
media_ranked AS (
    SELECT
        m.poi_id,
        m.media_id,
        m.blob_url,
        m.is_blessed,
        m.display_order,
        ROW_NUMBER() OVER (
            PARTITION BY m.poi_id, __MEDIA_IDENTITY_SQL__
            ORDER BY
                m._created_at ASC NULLS LAST,
                COALESCE(m.display_order, 2147483647) ASC,
                m.media_id ASC
        ) AS dedup_rn
    FROM poi_media m
    INNER JOIN v2_impacted_poi i
        ON i.poi_id = m.poi_id
    WHERE m.is_active = true
),
media_base AS (
    SELECT
        d.poi_id,
        d.media_id,
        d.blob_url,
        d.is_blessed,
        d.display_order,
        ROW_NUMBER() OVER (
            PARTITION BY d.poi_id
            ORDER BY
                CASE WHEN d.is_blessed = true THEN 0 ELSE 1 END,
                d.display_order ASC NULLS LAST,
                d.media_id ASC
        ) AS cover_rn
    FROM media_ranked d
    WHERE d.dedup_rn = 1
),
media_agg AS (
    SELECT
        mb.poi_id,
        to_json(
            named_struct(
                'cover',
                first(
                    CASE
                        WHEN mb.cover_rn = 1 THEN named_struct(
                            'media_id', mb.media_id,
                            'blob_url', mb.blob_url
                        )
                    END,
                    true
                ),
                'gallery',
                transform(
                    sort_array(
                        collect_list(
                            named_struct(
                                'sort_order', COALESCE(mb.display_order, 2147483647),
                                'media_id', mb.media_id,
                                'blob_url', mb.blob_url
                            )
                        )
                    ),
                    x -> named_struct(
                        'media_id', x.media_id,
                        'blob_url', x.blob_url
                    )
                ),
                'total_count',
                CAST(COUNT(*) AS INT)
            )
        ) AS poi_media_json
    FROM media_base mb
    GROUP BY mb.poi_id
),
review_agg AS (
    SELECT
        rv.poi_id,
        to_json(
            collect_list(
                named_struct(
                    'review_id', rv.review_id,
                    'language', rv.language,
                    'reviewer_name', rv.reviewer_name,
                    'rating_score', rv.rating_score,
                    'review_text', rv.review_text,
                    'review_date', rv.review_date,
                    'published_at', rv.published_at
                )
            )
        ) AS poi_review_json
    FROM poi_review rv
    INNER JOIN v2_impacted_poi i
        ON i.poi_id = rv.poi_id
    WHERE rv.is_visible = true
    GROUP BY rv.poi_id
)
SELECT
    e.poi_id,
    e.lang,

    COALESCE(ac.resolved_amenities_json, '{}') AS amenities_json,

    COALESCE(
        CASE
            WHEN a.poi_id IS NULL THEN NULL
            ELSE to_json(
                named_struct(
                    'address_line1', a.address_line1,
                    'address_line2', a.address_line2,
                    'ward', a.ward,
                    'district', a.district,
                    'city', a.city,
                    'state_province', a.state_province,
                    'country_code', a.country_code,
                    'postal_code', a.postal_code,
                    'short_address', a.short_address,
                    'full_address', a.full_address,
                    'lat', a.lat,
                    'lng', a.lng,
                    'geohash', a.geohash,
                    'h3_res8', a.h3_res8,
                    'plus_code', a.plus_code
                )
            )
        END,
        '{}'
    ) AS poi_address_json,

    COALESCE(
        CASE
            WHEN c.poi_id IS NULL THEN NULL
            ELSE to_json(
                named_struct(
                    'phone', c.phone,
                    'secondary_phones', c.secondary_phones,
                    'email', c.email,
                    'website_url', c.website_url,
                    'booking_url', c.booking_url,
                    'menu_url', c.menu_url,
                    'facebook_url', c.facebook_url,
                    'instagram_url', c.instagram_url,
                    'tiktok_url', c.tiktok_url
                )
            )
        END,
        '{}'
    ) AS poi_contact_json,

    COALESCE(
        CASE
            WHEN cp.poi_id IS NULL THEN NULL
            ELSE to_json(
                named_struct(
                    'locale', cp.locale,
                    'display_name', cp.display_name,
                    'description_short', cp.description_short,
                    'description_long', cp.description_long,
                    'service_bio', cp.service_bio,
                    'important_notice', cp.important_notice,
                    'price_range_text', cp.price_range_text,
                    'seo_title', cp.seo_title,
                    'seo_description', cp.seo_description,
                    'service_usps', cp.service_usps,
                    'tags_text', cp.tags_text,
                    'hours_weekday_text', cp.hours_weekday_text
                )
            )
        END,
        '{}'
    ) AS poi_content_json,

    COALESCE(
        ma.poi_media_json,
        '{"cover":null,"gallery":[],"total_count":0}'
    ) AS poi_media_json,

    COALESCE(
        CASE
            WHEN oh.poi_id IS NULL THEN NULL
            ELSE to_json(
                named_struct(
                    'timezone', oh.timezone,
                    'is_24_7', oh.is_24_7,
                    'open_now', oh.open_now,
                    'periods', oh.periods_json,
                    'holiday_exceptions', oh.holiday_exceptions_json
                )
            )
        END,
        '{}'
    ) AS poi_opening_hours_json,

    COALESCE(
        CASE
            WHEN pl.poi_id IS NULL THEN NULL
            ELSE to_json(
                named_struct(
                    'policy', pl.policy_json,
                    'source_updated_at', pl.source_updated_at
                )
            )
        END,
        '{}'
    ) AS poi_policy_json,

    COALESCE(
        CASE
            WHEN r.poi_id IS NULL THEN NULL
            ELSE to_json(
                named_struct(
                    'overall', r.overall,
                    'total_reviews', r.total_reviews,
                    'score_label', r.score_label,
                    'breakdown', r.breakdown_json
                )
            )
        END,
        '{}'
    ) AS poi_rating_json,

    COALESCE(ra.poi_review_json, '[]') AS poi_review_json,

    COALESCE(
        CASE
            WHEN pr.poi_id IS NULL THEN NULL
            ELSE to_json(
                named_struct(
                    'price',
                    named_struct(
                        'currency', pr.currency,
                        'price_level', pr.price_level,
                        'price_min', pr.price_min,
                        'price_max', pr.price_max
                    )
                )
            )
        END,
        '{}'
    ) AS poi_extra_json,

    e.source_id,
    e.event_id,
    e._created_at AS created_at,
    current_timestamp() AS updated_at,
    e.canonical_source AS ingestion_source,
    e.poi_group_id

FROM poi_entity e
INNER JOIN v2_impacted_poi i
    ON i.poi_id = e.poi_id
LEFT JOIN poi_address a
    ON a.poi_id = e.poi_id
LEFT JOIN poi_contact c
    ON c.poi_id = e.poi_id
LEFT JOIN content_pick cp
    ON cp.poi_id = e.poi_id
LEFT JOIN poi_amenity_current ac
    ON ac.poi_id = e.poi_id
LEFT JOIN poi_price pr
    ON pr.poi_id = e.poi_id
LEFT JOIN poi_opening_hours oh
    ON oh.poi_id = e.poi_id
LEFT JOIN poi_policy pl
    ON pl.poi_id = e.poi_id
LEFT JOIN poi_rating r
    ON r.poi_id = e.poi_id
LEFT JOIN media_agg ma
    ON ma.poi_id = e.poi_id
LEFT JOIN review_agg ra
    ON ra.poi_id = e.poi_id
WHERE e._is_current = true
"""
_detail_sql = _detail_sql.replace("__MEDIA_IDENTITY_SQL__", _MEDIA_IDENTITY_SQL)
spark.sql(_detail_sql)

# Fail early if the physical Gold target changes.
_expected_poi_detail_cols = {
    "poi_id","lang","amenities_json","poi_address_json","poi_contact_json",
    "poi_content_json","poi_media_json","poi_opening_hours_json",
    "poi_policy_json","poi_rating_json","poi_review_json","poi_extra_json",
    "source_id","event_id","created_at","updated_at","ingestion_source",
    "poi_group_id"
}
_poi_detail_target = "sgr_visitvn_dev.lh_vv_gold.dbo.poi_detail"
_poi_detail_actual = set(spark.table(_poi_detail_target).columns)
_poi_detail_missing = sorted(_expected_poi_detail_cols - _poi_detail_actual)

if _poi_detail_missing:
    raise RuntimeError(
        "NB40_TARGET_SCHEMA_MISMATCH poi_detail missing columns: "
        + str(_poi_detail_missing)
    )

print("NB40 poi_detail target schema preflight = OK")

spark.sql("""
MERGE INTO sgr_visitvn_dev.lh_vv_gold.dbo.poi_detail tgt
USING v2_gold_detail_src src
ON tgt.poi_id = src.poi_id

WHEN MATCHED THEN UPDATE SET
    tgt.lang                   = src.lang,
    tgt.amenities_json         = src.amenities_json,
    tgt.poi_address_json       = src.poi_address_json,
    tgt.poi_contact_json       = src.poi_contact_json,
    tgt.poi_content_json       = src.poi_content_json,
    tgt.poi_media_json         = src.poi_media_json,
    tgt.poi_opening_hours_json = src.poi_opening_hours_json,
    tgt.poi_policy_json        = src.poi_policy_json,
    tgt.poi_rating_json        = src.poi_rating_json,
    tgt.poi_review_json        = src.poi_review_json,
    tgt.poi_extra_json         = src.poi_extra_json,
    tgt.source_id              = src.source_id,
    tgt.event_id               = src.event_id,
    tgt.ingestion_source       = src.ingestion_source,
    tgt.poi_group_id           = src.poi_group_id,
    tgt.updated_at             = src.updated_at

WHEN NOT MATCHED THEN INSERT (
    poi_id,
    lang,
    amenities_json,
    poi_address_json,
    poi_contact_json,
    poi_content_json,
    poi_media_json,
    poi_opening_hours_json,
    poi_policy_json,
    poi_rating_json,
    poi_review_json,
    poi_extra_json,
    source_id,
    event_id,
    created_at,
    updated_at,
    ingestion_source,
    poi_group_id
)
VALUES (
    src.poi_id,
    src.lang,
    src.amenities_json,
    src.poi_address_json,
    src.poi_contact_json,
    src.poi_content_json,
    src.poi_media_json,
    src.poi_opening_hours_json,
    src.poi_policy_json,
    src.poi_rating_json,
    src.poi_review_json,
    src.poi_extra_json,
    src.source_id,
    src.event_id,
    src.created_at,
    src.updated_at,
    src.ingestion_source,
    src.poi_group_id
)
""")

# -----------------------------------------------------------------------------
# 2B. HISTORICAL JSON NOT-NULL REPAIR
#
# poi_detail is consumed by PostgreSQL targets where JSON payload columns are
# NOT NULL. Incremental MERGE only touches impacted POIs, therefore historical
# rows created by older notebook versions may still contain NULL.
#
# Contract:
#   object payloads  -> '{}'
#   poi_review_json  -> '[]'
#   poi_media_json   -> {"cover":null,"gallery":[],"total_count":0}
# -----------------------------------------------------------------------------
spark.sql("""
UPDATE sgr_visitvn_dev.lh_vv_gold.dbo.poi_detail
SET
    amenities_json          = COALESCE(amenities_json, '{}'),
    poi_address_json        = COALESCE(poi_address_json, '{}'),
    poi_contact_json        = COALESCE(poi_contact_json, '{}'),
    poi_content_json        = COALESCE(poi_content_json, '{}'),
    poi_media_json          = COALESCE(
        poi_media_json,
        '{"cover":null,"gallery":[],"total_count":0}'
    ),
    poi_opening_hours_json  = COALESCE(poi_opening_hours_json, '{}'),
    poi_policy_json         = COALESCE(poi_policy_json, '{}'),
    poi_rating_json         = COALESCE(poi_rating_json, '{}'),
    poi_review_json         = COALESCE(poi_review_json, '[]'),
    poi_extra_json          = COALESCE(poi_extra_json, '{}')
WHERE
       amenities_json         IS NULL
    OR poi_address_json       IS NULL
    OR poi_contact_json       IS NULL
    OR poi_content_json       IS NULL
    OR poi_media_json         IS NULL
    OR poi_opening_hours_json IS NULL
    OR poi_policy_json        IS NULL
    OR poi_rating_json        IS NULL
    OR poi_review_json        IS NULL
    OR poi_extra_json         IS NULL
""")

# -----------------------------------------------------------------------------
# Destination mapping is intentionally NOT recalculated here.
# v2_destination_resolved is the single source of truth and was already consumed
# by v2_gold_master_src above. This prevents a second resolver from overwriting
# ward+city / Phu Quoc results with the legacy ward+state_province logic.
# -----------------------------------------------------------------------------

# -----------------------------------------------------------------------------
# 3. Gold poi_rating, 4. Gold poi_review
# -----------------------------------------------------------------------------
# Gold poi_rating physical schema:
# poi_id, overall, total_reviews, score_label, sources_json, breakdown_json,
# sentiment_avg, source_id, event_id, updated_at, _batch_id
_expected_poi_rating_cols = {
    "poi_id","overall","total_reviews","score_label","sources_json",
    "breakdown_json","sentiment_avg","source_id","event_id","updated_at","_batch_id"
}
_poi_rating_target = "sgr_visitvn_dev.lh_vv_gold.dbo.poi_rating"
_poi_rating_actual = set(spark.table(_poi_rating_target).columns)
_poi_rating_missing = sorted(_expected_poi_rating_cols - _poi_rating_actual)
if _poi_rating_missing:
    raise RuntimeError(
        "NB40_TARGET_SCHEMA_MISMATCH poi_rating missing columns: "
        + str(_poi_rating_missing)
    )
print("NB40 poi_rating target schema preflight = OK")

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW v2_gold_rating_src AS
SELECT
    r.poi_id,
    r.overall,
    r.total_reviews,
    r.score_label,
    r.sources_json,
    r.breakdown_json,
    r.sentiment_avg,
    e.source_id,
    e.event_id,
    current_timestamp() AS updated_at,
    '{run_id}' AS _batch_id
FROM poi_rating r
INNER JOIN v2_impacted_poi i
    ON i.poi_id = r.poi_id
LEFT JOIN poi_entity e
    ON e.poi_id = r.poi_id
   AND e._is_current = true
""")

spark.sql("""
MERGE INTO sgr_visitvn_dev.lh_vv_gold.dbo.poi_rating tgt
USING v2_gold_rating_src src
ON tgt.poi_id = src.poi_id

WHEN MATCHED THEN UPDATE SET
    tgt.overall        = src.overall,
    tgt.total_reviews  = src.total_reviews,
    tgt.score_label    = src.score_label,
    tgt.sources_json   = src.sources_json,
    tgt.breakdown_json = src.breakdown_json,
    tgt.sentiment_avg  = src.sentiment_avg,
    tgt.source_id      = src.source_id,
    tgt.event_id       = src.event_id,
    tgt.updated_at     = src.updated_at,
    tgt._batch_id      = src._batch_id

WHEN NOT MATCHED THEN INSERT (
    poi_id,
    overall,
    total_reviews,
    score_label,
    sources_json,
    breakdown_json,
    sentiment_avg,
    source_id,
    event_id,
    updated_at,
    _batch_id
)
VALUES (
    src.poi_id,
    src.overall,
    src.total_reviews,
    src.score_label,
    src.sources_json,
    src.breakdown_json,
    src.sentiment_avg,
    src.source_id,
    src.event_id,
    src.updated_at,
    src._batch_id
)
""")

# -----------------------------------------------------------------------------
# 4. Gold poi_review
#
# Physical Gold schema confirmed from runtime:
# review_id, poi_id, source, source_review_id, lang, reviewer_name, rating,
# title, positive, negative, review_text, review_date, review_trip_type,
# sentiment_score, ranking_text, awards_json, response_text, responded_at,
# media_urls_json, helpful_votes, published_at, last_projected_at,
# source_id, event_id, _batch_id
# -----------------------------------------------------------------------------

_expected_poi_review_cols = {
    "review_id","poi_id","source","source_review_id","lang","reviewer_name",
    "rating","title","positive","negative","review_text","review_date",
    "review_trip_type","sentiment_score","ranking_text","awards_json",
    "response_text","responded_at","media_urls_json","helpful_votes",
    "published_at","last_projected_at","source_id","event_id","_batch_id"
}

_poi_review_target = "sgr_visitvn_dev.lh_vv_gold.dbo.poi_review"
_poi_review_actual = set(spark.table(_poi_review_target).columns)
_poi_review_missing = sorted(_expected_poi_review_cols - _poi_review_actual)

if _poi_review_missing:
    raise RuntimeError(
        "NB40_TARGET_SCHEMA_MISMATCH poi_review missing columns: "
        + str(_poi_review_missing)
    )

print("NB40 poi_review target schema preflight = OK")

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW v2_gold_review_src AS
SELECT
    r.review_id,
    r.poi_id,
    r.source_name AS source,
    r.source_review_id,
    r.language AS lang,
    r.reviewer_name,
    r.rating_score AS rating,
    r.review_title AS title,

    CAST(NULL AS STRING) AS positive,
    CAST(NULL AS STRING) AS negative,

    r.review_text,
    r.review_date,
    r.travel_type AS review_trip_type,

    CAST(NULL AS DOUBLE) AS sentiment_score,
    CAST(NULL AS STRING) AS ranking_text,
    CAST(NULL AS STRING) AS awards_json,

    r.response_text,
    r.responded_at,
    to_json(r.review_media_urls) AS media_urls_json,
    r.helpful_count AS helpful_votes,

    COALESCE(
        r.published_at,
        CAST(r.review_date AS TIMESTAMP),
        r._created_at
    ) AS published_at,

    current_timestamp() AS last_projected_at,
    e.source_id,
    e.event_id,
    '{run_id}' AS _batch_id

FROM poi_review r
INNER JOIN v2_impacted_poi i
    ON i.poi_id = r.poi_id
LEFT JOIN poi_entity e
    ON e.poi_id = r.poi_id
   AND e._is_current = true
WHERE r.is_visible = true
""")

spark.sql("""
MERGE INTO sgr_visitvn_dev.lh_vv_gold.dbo.poi_review tgt
USING v2_gold_review_src src
ON tgt.review_id = src.review_id

WHEN MATCHED THEN UPDATE SET
    tgt.poi_id            = src.poi_id,
    tgt.source            = src.source,
    tgt.source_review_id  = src.source_review_id,
    tgt.lang              = src.lang,
    tgt.reviewer_name     = src.reviewer_name,
    tgt.rating            = src.rating,
    tgt.title             = src.title,
    tgt.positive          = src.positive,
    tgt.negative          = src.negative,
    tgt.review_text       = src.review_text,
    tgt.review_date       = src.review_date,
    tgt.review_trip_type  = src.review_trip_type,
    tgt.sentiment_score   = src.sentiment_score,
    tgt.ranking_text      = src.ranking_text,
    tgt.awards_json       = src.awards_json,
    tgt.response_text     = src.response_text,
    tgt.responded_at      = src.responded_at,
    tgt.media_urls_json   = src.media_urls_json,
    tgt.helpful_votes     = src.helpful_votes,
    tgt.published_at      = src.published_at,
    tgt.last_projected_at = src.last_projected_at,
    tgt.source_id         = src.source_id,
    tgt.event_id          = src.event_id,
    tgt._batch_id         = src._batch_id

WHEN NOT MATCHED THEN INSERT (
    review_id,
    poi_id,
    source,
    source_review_id,
    lang,
    reviewer_name,
    rating,
    title,
    positive,
    negative,
    review_text,
    review_date,
    review_trip_type,
    sentiment_score,
    ranking_text,
    awards_json,
    response_text,
    responded_at,
    media_urls_json,
    helpful_votes,
    published_at,
    last_projected_at,
    source_id,
    event_id,
    _batch_id
)
VALUES (
    src.review_id,
    src.poi_id,
    src.source,
    src.source_review_id,
    src.lang,
    src.reviewer_name,
    src.rating,
    src.title,
    src.positive,
    src.negative,
    src.review_text,
    src.review_date,
    src.review_trip_type,
    src.sentiment_score,
    src.ranking_text,
    src.awards_json,
    src.response_text,
    src.responded_at,
    src.media_urls_json,
    src.helpful_votes,
    src.published_at,
    src.last_projected_at,
    src.source_id,
    src.event_id,
    src._batch_id
)
""")

# -----------------------------------------------------------------------------
# 5. Gold poi_enrichment_display — BUSINESS SOURCE OF TRUTH: Confluence 7A.2
# -----------------------------------------------------------------------------
#
# Contract:
#   1. Filter is_current = true
#   2. Deduplicate latest enriched_at per (poi_id, normalized lang)
#   3. Inner join active/current POI by poi_id
#   4. Normalize lang: vi/en/ko only; everything else -> en
#   5. display_category comes ONLY from final ref_business_sector
#      exact dictionary match; unknown -> other
#   6. Format duration / best_time / cost into language-specific display JSON
#   7. description direct-copy translated text
#   8. experiences keeps JSON shape; translated title/description are expected
#      to already be canonical in poi_enrichment; keywords remain unchanged
#   9. enrichment_job_id / validation_status are legacy Gold columns and are written NULL
#  10. display_id = <enrichment_id>_<target_poi_id>_<lang>_display
#
# IMPORTANT:
# - Do NOT use poi_entity.business_category for display_category.
# - ref_business_sector is the canonical source for display_category.
# - enrichment_job_id / validation_status are no longer used by business logic.
# -----------------------------------------------------------------------------

_expected_enrichment_display_cols = {
    "display_id","enrichment_id","poi_id","poi_name","lang","display_category",
    "description","time_to_spend","best_time","cost","experiences",
    "enrichment_job_id","validation_status","display_generated_at","is_current",
    "ingestion_source","created_at","updated_at"
}
_enrichment_display_target = "sgr_visitvn_dev.lh_vv_gold.dbo.poi_enrichment_display"
_enrichment_display_actual = set(spark.table(_enrichment_display_target).columns)
_enrichment_display_missing = sorted(
    _expected_enrichment_display_cols - _enrichment_display_actual
)
if _enrichment_display_missing:
    raise RuntimeError(
        "NB40_TARGET_SCHEMA_MISMATCH poi_enrichment_display missing columns: "
        + str(_enrichment_display_missing)
    )
print("NB40 poi_enrichment_display target schema preflight = OK")

# --------------------------------------------------------------
# 5.1 Canonical source + language normalization + latest dedupe
# --------------------------------------------------------------
spark.sql("""
CREATE OR REPLACE TEMP VIEW v2_7a2_enrichment_latest AS
WITH normalized AS (
    SELECT
        e.*,
        CASE
            WHEN lower(trim(e.lang)) IN ('vi','en','ko')
                THEN lower(trim(e.lang))
            ELSE 'en'
        END AS normalized_lang,

        CASE
            WHEN lower(trim(e.ref_business_sector)) IN (
                'accommodation',
                'f_n_b',
                'attraction',
                'tour_n_activities',
                'shopping_n_retail',
                'transport_route_base',
                'transport_on_demand',
                'vehicle_rental',
                'spa_n_wellness',
                'event',
                'other'
            )
            THEN lower(trim(e.ref_business_sector))
            ELSE 'other'
        END AS display_category_norm,

        ROW_NUMBER() OVER (
            PARTITION BY
                e.poi_id,
                CASE
                    WHEN lower(trim(e.lang)) IN ('vi','en','ko')
                        THEN lower(trim(e.lang))
                    ELSE 'en'
                END
            ORDER BY
                e.enriched_at DESC NULLS LAST,
                e._updated_at DESC NULLS LAST,
                e.enrichment_id DESC
        ) AS rn
    FROM poi_enrichment e
    INNER JOIN v2_impacted_poi i
        ON i.poi_id = e.poi_id
    WHERE e.is_current = true
)
SELECT *
FROM normalized
WHERE rn = 1
""")

# --------------------------------------------------------------
# 5.2 Active POI + localized formatter projection
# --------------------------------------------------------------
spark.sql("""
CREATE OR REPLACE TEMP VIEW v2_gold_enrichment_display_src AS
WITH src AS (
    SELECT
        e.*,
        p.canonical_source AS poi_ingestion_source
    FROM v2_7a2_enrichment_latest e
    INNER JOIN poi_entity p
        ON p.poi_id = e.poi_id
       AND p._is_current = true
       AND p.is_active = true
),
labels AS (
    SELECT
        s.*,

        -- Duration label
        CASE s.normalized_lang
            WHEN 'vi' THEN CASE s.display_category_norm
                WHEN 'accommodation'        THEN 'Thời gian lưu trú lý tưởng'
                WHEN 'f_n_b'                THEN 'Thời gian dùng bữa'
                WHEN 'attraction'           THEN 'Thời gian tham quan'
                WHEN 'tour_n_activities'    THEN 'Thời lượng hoạt động'
                WHEN 'shopping_n_retail'    THEN 'Thời gian mua sắm'
                WHEN 'transport_route_base' THEN 'Thời gian đưa đón'
                WHEN 'transport_on_demand'  THEN 'Thời gian thuê phổ biến'
                WHEN 'vehicle_rental'       THEN 'Thời gian thuê phổ biến'
                WHEN 'spa_n_wellness'       THEN 'Thời lượng liệu trình'
                WHEN 'event'                THEN 'Thời lượng chương trình'
                ELSE 'Thời lượng trải nghiệm'
            END
            WHEN 'ko' THEN CASE s.display_category_norm
                WHEN 'accommodation'        THEN '권장 숙박 시간'
                WHEN 'f_n_b'                THEN '식사 소요 시간'
                WHEN 'attraction'           THEN '관람 소요 시간'
                WHEN 'tour_n_activities'    THEN '활동 소요 시간'
                WHEN 'shopping_n_retail'    THEN '쇼핑 소요 시간'
                WHEN 'transport_route_base' THEN '이동 소요 시간'
                WHEN 'transport_on_demand'  THEN '일반 대여 시간'
                WHEN 'vehicle_rental'       THEN '일반 대여 시간'
                WHEN 'spa_n_wellness'       THEN '관리 소요 시간'
                WHEN 'event'                THEN '행사 소요 시간'
                ELSE '체험 소요 시간'
            END
            ELSE CASE s.display_category_norm
                WHEN 'accommodation'        THEN 'Ideal stay duration'
                WHEN 'f_n_b'                THEN 'Meal duration'
                WHEN 'attraction'           THEN 'Visit duration'
                WHEN 'tour_n_activities'    THEN 'Activity duration'
                WHEN 'shopping_n_retail'    THEN 'Shopping duration'
                WHEN 'transport_route_base' THEN 'Transfer duration'
                WHEN 'transport_on_demand'  THEN 'Common rental duration'
                WHEN 'vehicle_rental'       THEN 'Common rental duration'
                WHEN 'spa_n_wellness'       THEN 'Treatment duration'
                WHEN 'event'                THEN 'Event duration'
                ELSE 'Experience duration'
            END
        END AS duration_label,

        -- Best-time label
        CASE s.normalized_lang
            WHEN 'vi' THEN CASE s.display_category_norm
                WHEN 'accommodation'        THEN 'Mùa lưu trú đẹp nhất'
                WHEN 'f_n_b'                THEN 'Thời điểm nên ghé'
                WHEN 'attraction'           THEN 'Mùa đẹp nhất'
                WHEN 'tour_n_activities'    THEN 'Thời điểm phù hợp'
                WHEN 'shopping_n_retail'    THEN 'Thời điểm nên ghé'
                WHEN 'transport_route_base' THEN 'Thời điểm phù hợp'
                WHEN 'transport_on_demand'  THEN 'Thời điểm nên thuê'
                WHEN 'vehicle_rental'       THEN 'Thời điểm nên thuê'
                WHEN 'spa_n_wellness'       THEN 'Thời điểm phù hợp'
                WHEN 'event'                THEN 'Thời điểm nên xem'
                ELSE 'Thời điểm phù hợp'
            END
            WHEN 'ko' THEN CASE s.display_category_norm
                WHEN 'accommodation'        THEN '추천 숙박 시즌'
                WHEN 'f_n_b'                THEN '방문하기 좋은 시간'
                WHEN 'attraction'           THEN '추천 시즌'
                WHEN 'tour_n_activities'    THEN '적합한 시기'
                WHEN 'shopping_n_retail'    THEN '방문하기 좋은 시간'
                WHEN 'transport_route_base' THEN '적합한 시간'
                WHEN 'transport_on_demand'  THEN '대여하기 좋은 시기'
                WHEN 'vehicle_rental'       THEN '대여하기 좋은 시기'
                WHEN 'spa_n_wellness'       THEN '적합한 시간'
                WHEN 'event'                THEN '참석하기 좋은 시간'
                ELSE '적합한 시간'
            END
            ELSE CASE s.display_category_norm
                WHEN 'accommodation'        THEN 'Best stay season'
                WHEN 'f_n_b'                THEN 'Best time to visit'
                WHEN 'attraction'           THEN 'Best season'
                WHEN 'tour_n_activities'    THEN 'Suitable time'
                WHEN 'shopping_n_retail'    THEN 'Best time to visit'
                WHEN 'transport_route_base' THEN 'Suitable time'
                WHEN 'transport_on_demand'  THEN 'Best rental time'
                WHEN 'vehicle_rental'       THEN 'Best rental time'
                WHEN 'spa_n_wellness'       THEN 'Suitable time'
                WHEN 'event'                THEN 'Best time to attend'
                ELSE 'Best time'
            END
        END AS best_time_label,

        -- Cost label
        CASE s.normalized_lang
            WHEN 'vi' THEN CASE s.display_category_norm
                WHEN 'accommodation'        THEN 'Giá phòng tham khảo'
                WHEN 'f_n_b'                THEN 'Mức giá tham khảo'
                WHEN 'attraction'           THEN 'Chi phí tham quan'
                WHEN 'tour_n_activities'    THEN 'Chi phí tham khảo'
                WHEN 'shopping_n_retail'    THEN 'Chi tiêu tham khảo'
                WHEN 'transport_route_base' THEN 'Chi phí đưa đón'
                WHEN 'transport_on_demand'  THEN 'Giá thuê tham khảo'
                WHEN 'vehicle_rental'       THEN 'Giá thuê tham khảo'
                WHEN 'spa_n_wellness'       THEN 'Giá dịch vụ tham khảo'
                WHEN 'event'                THEN 'Giá vé tham khảo'
                ELSE 'Chi phí tham khảo'
            END
            WHEN 'ko' THEN CASE s.display_category_norm
                WHEN 'accommodation'        THEN '참고 객실 요금'
                WHEN 'f_n_b'                THEN '참고 가격'
                WHEN 'attraction'           THEN '관람 비용'
                WHEN 'tour_n_activities'    THEN '참고 비용'
                WHEN 'shopping_n_retail'    THEN '참고 지출'
                WHEN 'transport_route_base' THEN '이동 비용'
                WHEN 'transport_on_demand'  THEN '참고 대여 요금'
                WHEN 'vehicle_rental'       THEN '참고 대여 요금'
                WHEN 'spa_n_wellness'       THEN '참고 서비스 요금'
                WHEN 'event'                THEN '참고 티켓 가격'
                ELSE '참고 비용'
            END
            ELSE CASE s.display_category_norm
                WHEN 'accommodation'        THEN 'Reference room rate'
                WHEN 'f_n_b'                THEN 'Reference price'
                WHEN 'attraction'           THEN 'Visit cost'
                WHEN 'tour_n_activities'    THEN 'Reference cost'
                WHEN 'shopping_n_retail'    THEN 'Reference spend'
                WHEN 'transport_route_base' THEN 'Transfer cost'
                WHEN 'transport_on_demand'  THEN 'Reference rental rate'
                WHEN 'vehicle_rental'       THEN 'Reference rental rate'
                WHEN 'spa_n_wellness'       THEN 'Reference service rate'
                WHEN 'event'                THEN 'Reference ticket price'
                ELSE 'Reference cost'
            END
        END AS cost_label
    FROM src s
),
formatted AS (
    SELECT
        l.*,

        -- Duration formatter from 7A.2.
        CASE
            WHEN l.time_to_spend_minutes IS NULL OR l.time_to_spend_minutes <= 0
            THEN CASE l.normalized_lang
                WHEN 'vi' THEN 'Không cố định'
                WHEN 'ko' THEN '유동적'
                ELSE 'Flexible'
            END

            WHEN l.display_category_norm = 'accommodation'
            THEN CASE l.normalized_lang
                WHEN 'vi' THEN concat(
                    CAST(ceil(l.time_to_spend_minutes / 1440.0) AS BIGINT),
                    ' ngày'
                )
                WHEN 'ko' THEN concat(
                    CAST(ceil(l.time_to_spend_minutes / 1440.0) AS BIGINT),
                    '일'
                )
                ELSE concat(
                    CAST(ceil(l.time_to_spend_minutes / 1440.0) AS BIGINT),
                    CASE
                        WHEN ceil(l.time_to_spend_minutes / 1440.0) = 1 THEN ' day'
                        ELSE ' days'
                    END
                )
            END

            WHEN l.display_category_norm = 'tour_n_activities'
             AND l.time_to_spend_minutes >= 420
            THEN CASE l.normalized_lang
                WHEN 'vi' THEN 'Cả ngày'
                WHEN 'ko' THEN '종일'
                ELSE 'Full day'
            END

            WHEN l.time_to_spend_minutes < 60
            THEN CASE l.normalized_lang
                WHEN 'vi' THEN concat(l.time_to_spend_minutes, ' phút')
                WHEN 'ko' THEN concat(l.time_to_spend_minutes, '분')
                ELSE concat(
                    l.time_to_spend_minutes,
                    CASE WHEN l.time_to_spend_minutes = 1 THEN ' minute' ELSE ' minutes' END
                )
            END

            ELSE CASE l.normalized_lang
                WHEN 'vi' THEN concat(
                    CAST(floor(l.time_to_spend_minutes / 60) AS BIGINT),
                    ' tiếng',
                    CASE
                        WHEN pmod(l.time_to_spend_minutes, 60) > 0
                        THEN concat(' ', pmod(l.time_to_spend_minutes, 60), ' phút')
                        ELSE ''
                    END
                )
                WHEN 'ko' THEN concat(
                    CAST(floor(l.time_to_spend_minutes / 60) AS BIGINT),
                    '시간',
                    CASE
                        WHEN pmod(l.time_to_spend_minutes, 60) > 0
                        THEN concat(' ', pmod(l.time_to_spend_minutes, 60), '분')
                        ELSE ''
                    END
                )
                ELSE concat(
                    CAST(floor(l.time_to_spend_minutes / 60) AS BIGINT),
                    CASE
                        WHEN floor(l.time_to_spend_minutes / 60) = 1 THEN ' hour'
                        ELSE ' hours'
                    END,
                    CASE
                        WHEN pmod(l.time_to_spend_minutes, 60) > 0
                        THEN concat(
                            ' ',
                            pmod(l.time_to_spend_minutes, 60),
                            CASE
                                WHEN pmod(l.time_to_spend_minutes, 60) = 1
                                THEN ' minute'
                                ELSE ' minutes'
                            END
                        )
                        ELSE ''
                    END
                )
            END
        END AS duration_text,

        -- Best-time formatter from 7A.2:
        -- month_range is shown only when both months are valid 1..12.
        CASE
            WHEN lower(trim(l.best_time_type)) = 'month_range'
             AND l.best_time_from_month BETWEEN 1 AND 12
             AND l.best_time_to_month BETWEEN 1 AND 12
            THEN CASE l.normalized_lang
                WHEN 'vi' THEN concat(
                    'Tháng ', l.best_time_from_month,
                    ' - Tháng ', l.best_time_to_month
                )
                WHEN 'ko' THEN concat(
                    l.best_time_from_month, '월 - ',
                    l.best_time_to_month, '월'
                )
                ELSE concat(
                    element_at(
                        array(
                            'January','February','March','April','May','June',
                            'July','August','September','October','November','December'
                        ),
                        l.best_time_from_month
                    ),
                    ' - ',
                    element_at(
                        array(
                            'January','February','March','April','May','June',
                            'July','August','September','October','November','December'
                        ),
                        l.best_time_to_month
                    )
                )
            END
            ELSE CASE l.normalized_lang
                WHEN 'vi' THEN 'Quanh năm'
                WHEN 'ko' THEN '연중'
                ELSE 'Year round'
            END
        END AS best_time_text,

        -- Cost formatter: round to nearest 1,000 VND; NULL stays SQL NULL.
        CASE
            WHEN l.cost_vnd IS NULL THEN NULL
            ELSE CASE l.normalized_lang
                WHEN 'vi' THEN concat(
                    'Từ ',
                    format_number(round(l.cost_vnd / 1000.0) * 1000, 0),
                    'đ'
                )
                WHEN 'ko' THEN concat(
                    format_number(round(l.cost_vnd / 1000.0) * 1000, 0),
                    '동부터'
                )
                ELSE concat(
                    'From ',
                    format_number(round(l.cost_vnd / 1000.0) * 1000, 0),
                    ' VND'
                )
            END
        END AS cost_text
    FROM labels l
)
SELECT
    concat(
        f.enrichment_id,
        '_',
        f.poi_id,
        '_',
        f.normalized_lang,
        '_display'
    ) AS display_id,

    f.enrichment_id,
    f.poi_id,
    f.poi_name,
    f.normalized_lang AS lang,
    f.display_category_norm AS display_category,
    f.description,

    to_json(
        map(
            f.duration_label,
            f.duration_text
        )
    ) AS time_to_spend,

    to_json(
        map(
            f.best_time_label,
            f.best_time_text
        )
    ) AS best_time,

    CASE
        WHEN f.cost_text IS NULL THEN NULL
        ELSE to_json(
            map(
                f.cost_label,
                f.cost_text
            )
        )
    END AS cost,

    f.experiences_json AS experiences,

    CAST(NULL AS STRING) AS enrichment_job_id,
    CAST(NULL AS STRING) AS validation_status,

    COALESCE(f.enriched_at, f._updated_at, current_timestamp())
        AS display_generated_at,

    true AS is_current,

    COALESCE(f.poi_ingestion_source, 'ENRICHMENT') AS ingestion_source,

    COALESCE(f._created_at, f.enriched_at, current_timestamp()) AS created_at,
    current_timestamp() AS updated_at
FROM formatted f
""")

# --------------------------------------------------------------
# 5.3 Defensive checks
# --------------------------------------------------------------
_dup_display_id = spark.sql("""
SELECT COUNT(*) AS c
FROM (
    SELECT display_id
    FROM v2_gold_enrichment_display_src
    GROUP BY display_id
    HAVING COUNT(*) > 1
) x
""").first()["c"]

if _dup_display_id > 0:
    raise RuntimeError(
        f"NB40_7A2_DUPLICATE_DISPLAY_ID duplicate_count={_dup_display_id}"
    )

_dup_poi = spark.sql("""
SELECT COUNT(*) AS c
FROM (
    SELECT poi_id
    FROM v2_gold_enrichment_display_src
    GROUP BY poi_id
    HAVING COUNT(*) > 1
) x
""").first()["c"]

if _dup_poi > 0:
    raise RuntimeError(
        f"NB40_7A2_DUPLICATE_POI duplicate_count={_dup_poi}"
    )

# --------------------------------------------------------------
# 5.4 Remove stale/history rows for impacted POIs.
# Business requirement: poi_enrichment_display keeps exactly ONE row per poi_id,
# and that surviving row must be is_current = true.
# --------------------------------------------------------------
spark.sql("""
MERGE INTO sgr_visitvn_dev.lh_vv_gold.dbo.poi_enrichment_display tgt
USING (
    SELECT DISTINCT poi_id, display_id
    FROM v2_gold_enrichment_display_src
) src
ON tgt.poi_id = src.poi_id
AND tgt.display_id <> src.display_id

WHEN MATCHED THEN DELETE
""")

# --------------------------------------------------------------
# 5.5 Idempotent display MERGE
# --------------------------------------------------------------
spark.sql("""
MERGE INTO sgr_visitvn_dev.lh_vv_gold.dbo.poi_enrichment_display tgt
USING v2_gold_enrichment_display_src src
ON tgt.display_id = src.display_id

WHEN MATCHED THEN UPDATE SET
    tgt.enrichment_id        = src.enrichment_id,
    tgt.poi_id               = src.poi_id,
    tgt.poi_name             = src.poi_name,
    tgt.lang                 = src.lang,
    tgt.display_category     = src.display_category,
    tgt.description          = src.description,
    tgt.time_to_spend        = src.time_to_spend,
    tgt.best_time            = src.best_time,
    tgt.cost                 = src.cost,
    tgt.experiences          = src.experiences,
    tgt.enrichment_job_id    = src.enrichment_job_id,
    tgt.validation_status    = src.validation_status,
    tgt.display_generated_at = src.display_generated_at,
    tgt.is_current           = src.is_current,
    tgt.ingestion_source     = src.ingestion_source,
    tgt.updated_at           = src.updated_at

WHEN NOT MATCHED THEN INSERT (
    display_id,
    enrichment_id,
    poi_id,
    poi_name,
    lang,
    display_category,
    description,
    time_to_spend,
    best_time,
    cost,
    experiences,
    enrichment_job_id,
    validation_status,
    display_generated_at,
    is_current,
    ingestion_source,
    created_at,
    updated_at
)
VALUES (
    src.display_id,
    src.enrichment_id,
    src.poi_id,
    src.poi_name,
    src.lang,
    src.display_category,
    src.description,
    src.time_to_spend,
    src.best_time,
    src.cost,
    src.experiences,
    src.enrichment_job_id,
    src.validation_status,
    src.display_generated_at,
    src.is_current,
    src.ingestion_source,
    src.created_at,
    src.updated_at
)
""")

# --------------------------------------------------------------
# 5.6 7A.2 post-merge invariants
# --------------------------------------------------------------
_missing_display = spark.sql("""
SELECT COUNT(*) AS c
FROM v2_gold_enrichment_display_src src
LEFT ANTI JOIN sgr_visitvn_dev.lh_vv_gold.dbo.poi_enrichment_display tgt
    ON tgt.display_id = src.display_id
   AND tgt.is_current = true
""").first()["c"]

if _missing_display > 0:
    raise RuntimeError(
        f"NB40_7A2_DISPLAY_INVARIANT_FAILED missing_current_display={_missing_display}"
    )

_bad_display_cardinality = spark.sql("""
SELECT COUNT(*) AS c
FROM (
    SELECT
        poi_id,
        COUNT(*) AS total_rows,
        SUM(CASE WHEN is_current = true THEN 1 ELSE 0 END) AS current_rows
    FROM sgr_visitvn_dev.lh_vv_gold.dbo.poi_enrichment_display
    GROUP BY poi_id
    HAVING COUNT(*) <> 1
        OR SUM(CASE WHEN is_current = true THEN 1 ELSE 0 END) <> 1
) x
""").first()["c"]

if _bad_display_cardinality > 0:
    raise RuntimeError(
        f"NB40_7A2_POI_CARDINALITY_FAILED invalid_poi_count={_bad_display_cardinality}"
    )

_display_count = spark.sql(
    "SELECT COUNT(*) AS c FROM v2_gold_enrichment_display_src"
).first()["c"]

print(f"NB40 7A.2 enrichment_display projected = {_display_count}")

# -----------------------------------------------------------------------------
# Hard invariant: a publishable impacted Silver entity with source identity must
# be materialized in Gold poi_master before NB40 succeeds.
# -----------------------------------------------------------------------------
missing_gold_master = spark.sql("""
SELECT COUNT(*) AS c
FROM (
    SELECT DISTINCT e.poi_id
    FROM poi_entity e
    INNER JOIN v2_impacted_poi i ON i.poi_id=e.poi_id
    INNER JOIN poi_source_map sm ON sm.poi_id=e.poi_id
    LEFT ANTI JOIN sgr_visitvn_dev.lh_vv_gold.dbo.poi_master m ON m.poi_id=e.poi_id
    WHERE e._is_current=true
      AND e.poi_name IS NOT NULL
) x
""").first()["c"]

master_src_count = spark.sql("SELECT COUNT(*) AS c FROM v2_gold_master_src").first()["c"]
impacted_count = spark.sql("SELECT COUNT(*) AS c FROM v2_impacted_poi").first()["c"]

print(f"NB40 impacted_count           = {impacted_count}")
print(f"NB40 master_src_count         = {master_src_count}")
print(f"NB40 missing_gold_master      = {missing_gold_master}")

if missing_gold_master > 0:
    raise RuntimeError(
        f"NB40_GOLD_MASTER_INVARIANT_FAILED missing_gold_master={missing_gold_master}"
    )

notebookutils.notebook.exit("SUCCESS")


In [ ]:
# # NB40 V7 - stable baseline + Fabric-safe media dedup
# # - Based on the last stable MEDIA_CONTRACT_FIXED notebook, not V4/V5/V6.
# # - Preserves the established namespace/execution contract.
# # - Removes correlated media aggregation and deduplicates in CTEs.

# # Fabric Notebook: NB_40_GOLD_POI_SERVING
# # READ : Silver (+ existing Gold destination reference tables)
# # WRITE: Gold poi_master, poi_detail, poi_rating, poi_review, poi_enrichment_display

# run_id = globals().get("run_id", "DEV_RUN")
# STAGE = "_stage_poi_language"

# # Only impacted logical POIs are rebuilt.
# spark.sql(f"""
# CREATE OR REPLACE TEMP VIEW v2_impacted_poi AS
# SELECT DISTINCT poi_id,poi_group_id,lang FROM {STAGE} WHERE run_id='{run_id}'
# """)

# # Optional migration columns
# for tbl in ["sgr_visitvn_dev.lh_vv_gold.dbo.poi_master", "sgr_visitvn_dev.lh_vv_gold.dbo.poi_detail"]:
#     try:
#         spark.sql(f"ALTER TABLE {tbl} ADD COLUMNS (poi_group_id STRING)")
#     except Exception:
#         pass

# # -----------------------------------------------------------------------------
# # 7A.2 BUSINESS CONTRACT PREFLIGHT
# #
# # Business source of truth:
# # poi_enrichment -> poi_enrichment_display
# #
# # NB40 intentionally does NOT infer these canonical fields from legacy columns.
# # If they are absent, NB30 / poi_enrichment schema must be aligned first.
# # -----------------------------------------------------------------------------
# _REQUIRED_7A2_POI_ENRICHMENT_COLS = {
#     "enrichment_id",
#     "poi_id",
#     "poi_name",
#     "lang",
#     "ref_business_sector",
#     "description",
#     "time_to_spend_minutes",
#     "best_time_type",
#     "best_time_from_month",
#     "best_time_to_month",
#     "cost_vnd",
#     "experiences_json",
#     "enriched_at",
#     "is_current",
#     "_created_at",
#     "_updated_at"
# }

# _actual_enrichment_cols = set(spark.table("poi_enrichment").columns)
# _missing_7a2_cols = sorted(
#     _REQUIRED_7A2_POI_ENRICHMENT_COLS - _actual_enrichment_cols
# )

# if _missing_7a2_cols:
#     raise RuntimeError(
#         "NB40_7A2_SOURCE_SCHEMA_MISMATCH: poi_enrichment is missing canonical "
#         f"business fields required by 7A.2: {_missing_7a2_cols}. "
#         "Required canonical 7A.2 source fields are missing. "
#         "Align NB30/poi_enrichment first."
#     )

# print("NB40 7A.2 source schema preflight = OK")

# # -----------------------------------------------------------------------------
# # NB40 PHYSICAL-SCHEMA PREFLIGHT
# # Fail before any Gold MERGE if a target schema drifts.
# # -----------------------------------------------------------------------------
# _TARGET_SCHEMAS = {
#     "sgr_visitvn_dev.lh_vv_gold.dbo.poi_master": {
#         "poi_id","lang","slug","poi_name","business_sector","business_category",
#         "subcategory_tags","operating_status","is_active","is_verified",
#         "short_description","short_address","district","rating_overall","rating_count",
#         "destination_ids","nearby_pois","destination_names","created_at","updated_at",
#         "version","source_id","event_id","ref_type","ref_business_sector",
#         "ingestion_source","partner_id","is_bookable","poi_group_id"
#     },
#     "sgr_visitvn_dev.lh_vv_gold.dbo.poi_detail": {
#         "poi_id","lang","amenities_json","poi_address_json","poi_contact_json",
#         "poi_content_json","poi_media_json","poi_opening_hours_json","poi_policy_json",
#         "poi_rating_json","poi_review_json","poi_extra_json","source_id","event_id",
#         "created_at","updated_at","ingestion_source","poi_group_id"
#     },
#     "sgr_visitvn_dev.lh_vv_gold.dbo.poi_rating": {
#         "poi_id","overall","total_reviews","score_label","sources_json","breakdown_json",
#         "sentiment_avg","source_id","event_id","updated_at","_batch_id"
#     },
#     "sgr_visitvn_dev.lh_vv_gold.dbo.poi_review": {
#         "review_id","poi_id","source","source_review_id","lang","reviewer_name","rating",
#         "title","positive","negative","review_text","review_date","review_trip_type",
#         "sentiment_score","ranking_text","awards_json","response_text","responded_at",
#         "media_urls_json","helpful_votes","published_at","last_projected_at",
#         "source_id","event_id","_batch_id"
#     },
#     "sgr_visitvn_dev.lh_vv_gold.dbo.poi_enrichment_display": {
#         "display_id","enrichment_id","poi_id","poi_name","lang","display_category",
#         "description","time_to_spend","best_time","cost","experiences",
#         "enrichment_job_id","validation_status","display_generated_at","is_current",
#         "ingestion_source","created_at","updated_at"
#     }
# }

# for _tbl, _expected in _TARGET_SCHEMAS.items():
#     _actual = set(spark.table(_tbl).columns)
#     _missing = sorted(_expected - _actual)
#     _unexpected = sorted(_actual - _expected)
#     if _missing:
#         raise RuntimeError(
#             f"NB40_SCHEMA_PREFLIGHT_FAILED table={_tbl} "
#             f"missing={_missing} unexpected={_unexpected}"
#         )
#     print(
#         f"NB40 schema preflight OK: {_tbl} "
#         f"(columns={len(_actual)}, unexpected={_unexpected})"
#     )

# # 1. poi_master - impacted-only and language-specific.
# # NOTE: normalize_vn() is intentionally NOT used because it is not a built-in
# # Spark SQL routine in Fabric. Province matching below uses only built-in
# # lower/trim/regexp_replace/translate functions for Vietnamese diacritic removal.
# spark.sql("""
# CREATE OR REPLACE TEMP VIEW v2_gold_master_src AS
# WITH entity AS (
#  SELECT e.* FROM poi_entity e JOIN v2_impacted_poi i ON i.poi_id=e.poi_id
#  WHERE e._is_current=true AND e.poi_name IS NOT NULL
# ), content_pick AS (
#  SELECT * FROM (
#    SELECT c.*,e.lang,
#      ROW_NUMBER() OVER(PARTITION BY c.poi_id ORDER BY CASE WHEN c.locale=e.lang THEN 0 WHEN c.locale='vi' THEN 1 WHEN c.locale='en' THEN 2 ELSE 3 END) rn
#    FROM poi_content c JOIN entity e ON e.poi_id=c.poi_id
#  ) WHERE rn=1
# ), dest_mapping AS (
#  SELECT DISTINCT ward, lower(trim(regexp_replace(translate(lower(coalesce(province_name,'')), 'àáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ', 'aaaaaaaaaaaaaaaaaeeeeeeeeeeeiiiiiooooooooooooooooouuuuuuuuuuuyyyyyd'), '\\s+', ' '))) province_name_nodiac,dest_id,dest_name
#  FROM sgr_visitvn_dev.lh_vv_gold.dbo.destination_ward_mapping WHERE ward IS NOT NULL AND province_name IS NOT NULL
# ), projected AS (
#  SELECT e.poi_id,e.poi_group_id,e.lang,e.slug,e.poi_name,e.business_sector,e.business_category,
#    to_json(e.subcategory_tags) subcategory_tags,e.operating_status,e.is_active,e.is_verified,
#    c.description_short short_description,COALESCE(a.short_address,a.full_address) short_address,a.district,
#    r.overall rating_overall,COALESCE(r.total_reviews,0) rating_count,
#    CASE WHEN dm.dest_id IS NOT NULL THEN to_json(array(dm.dest_id)) END destination_ids,
#    CASE WHEN dm.dest_name IS NOT NULL THEN to_json(array(dm.dest_name)) END destination_names,
#    e.canonical_source ingestion_source,e.partner_id,e.source_id,e.event_id,e._created_at created_at,e._updated_at updated_at,
#    CAST(NULL AS INT) version,dm.dest_id _dest_id_for_join
#  FROM entity e
#  INNER JOIN poi_source_map sm ON sm.poi_id=e.poi_id
#  LEFT JOIN poi_address a ON a.poi_id=e.poi_id
#  LEFT JOIN poi_rating r ON r.poi_id=e.poi_id
#  LEFT JOIN content_pick c ON c.poi_id=e.poi_id
#  LEFT JOIN dest_mapping dm ON dm.ward=a.ward AND dm.province_name_nodiac=lower(trim(regexp_replace(translate(lower(coalesce(a.state_province,'')), 'àáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ', 'aaaaaaaaaaaaaaaaaeeeeeeeeeeeiiiiiooooooooooooooooouuuuuuuuuuuyyyyyd'), '\\s+', ' ')))
# ), nearby AS (
#  SELECT p.poi_id,to_json(collect_list(a.poi_id)) nearby_pois
#  FROM projected p JOIN anchor_poi_map a ON a.destination_id=p._dest_id_for_join AND a.poi_id<>p.poi_id
#  GROUP BY p.poi_id
# )
# SELECT p.*,n.nearby_pois FROM projected p LEFT JOIN nearby n ON n.poi_id=p.poi_id
# """)

# spark.sql("""
# MERGE INTO sgr_visitvn_dev.lh_vv_gold.dbo.poi_master tgt
# USING v2_gold_master_src src ON tgt.poi_id=src.poi_id
# WHEN MATCHED THEN UPDATE SET
#  tgt.poi_group_id=src.poi_group_id,tgt.lang=src.lang,tgt.slug=src.slug,tgt.poi_name=src.poi_name,
#  tgt.business_sector=src.business_sector,tgt.business_category=src.business_category,tgt.subcategory_tags=src.subcategory_tags,
#  tgt.operating_status=src.operating_status,tgt.is_active=src.is_active,tgt.is_verified=src.is_verified,
#  tgt.short_description=src.short_description,tgt.short_address=src.short_address,tgt.district=src.district,
#  tgt.rating_overall=src.rating_overall,tgt.rating_count=src.rating_count,
#  tgt.destination_ids=COALESCE(src.destination_ids,tgt.destination_ids),tgt.destination_names=COALESCE(src.destination_names,tgt.destination_names),
#  tgt.nearby_pois=COALESCE(src.nearby_pois,tgt.nearby_pois),tgt.ingestion_source=src.ingestion_source,
#  tgt.partner_id=src.partner_id,tgt.source_id=src.source_id,tgt.event_id=src.event_id,tgt.updated_at=src.updated_at
# WHEN NOT MATCHED THEN INSERT(
#  poi_id,poi_group_id,lang,slug,poi_name,business_sector,business_category,subcategory_tags,operating_status,is_active,is_verified,
#  short_description,short_address,district,rating_overall,rating_count,destination_ids,destination_names,nearby_pois,ingestion_source,
#  partner_id,source_id,event_id,created_at,updated_at,version,is_bookable
# ) VALUES(
#  src.poi_id,src.poi_group_id,src.lang,src.slug,src.poi_name,src.business_sector,src.business_category,src.subcategory_tags,
#  src.operating_status,src.is_active,src.is_verified,src.short_description,src.short_address,src.district,src.rating_overall,src.rating_count,
#  src.destination_ids,src.destination_names,src.nearby_pois,src.ingestion_source,src.partner_id,src.source_id,src.event_id,
#  src.created_at,src.updated_at,COALESCE(src.version,0),false
# )
# """)

# # -----------------------------------------------------------------------------
# # MEDIA DEDUP CONTRACT - downstream consumer
# #
# # NB20 is the ONLY authoritative writer of media business identity:
# #   media_dedup_key = SHA256(source_name|source_id|media_id)
# #
# # NB40 must not recompute the key from photo_api_uri/original_url. It only
# # consumes the persisted key and performs a defensive POI-scoped dedup before
# # building poi_detail.poi_media_json.
# #
# # Historical rows that predate media_dedup_key fall back to media_id only.
# # This fallback preserves historical rows without redefining original_url as a
# # business identity.
# # -----------------------------------------------------------------------------
# _media_cols = set(spark.table("poi_media").columns)
# if "media_dedup_key" not in _media_cols:
#     raise RuntimeError(
#         "NB40_MEDIA_SCHEMA_MISMATCH: poi_media.media_dedup_key is required. "
#         "Run the updated NB20 before NB40."
#     )

# _MEDIA_IDENTITY_SQL = (
#     "COALESCE(NULLIF(TRIM(m.media_dedup_key), ''), CONCAT('LEGACY_ID:', m.media_id))"
# )

# print(
#     "NB40 media contract = poi_id + persisted media_dedup_key "
#     "(NB20 authoritative; media_id fallback only for historical rows)"
# )

# # 2. poi_detail - packed API document, impacted only.
# # Physical Gold schema:
# #   poi_id, lang, amenities_json, poi_address_json, poi_contact_json,
# #   poi_content_json, poi_media_json, poi_opening_hours_json, poi_policy_json,
# #   poi_rating_json, poi_review_json, poi_extra_json, source_id, event_id,
# #   created_at, updated_at, ingestion_source, poi_group_id
# #
# # Optional Silver components are LEFT JOINed. They must never remove a POI.
# # Gold poi_detail JSON contract:
# #   object JSON columns -> '{}' when source data is absent
# #   poi_review_json     -> '[]' when source data is absent
# #   poi_media_json      -> {"cover":null,"gallery":[],"total_count":0}
# #
# # poi_media_json business contract:
# # {
# #   "cover":  {"media_id": "...", "blob_url": "..."} | null,
# #   "gallery": [{"media_id": "...", "blob_url": "..."}],
# #   "total_count": N
# # }
# # Cover priority: is_blessed=true -> display_order ASC -> media_id ASC.
# # Gallery contains all active media, including the selected cover.
# # This prevents downstream PostgreSQL NOT NULL violations.
# # Amenities parity with the legacy pipeline:
# #   poi_amenity_current.resolved_amenities_json -> poi_detail.amenities_json
# #   fallback '{}' when the amenity row/payload is missing.
# # Content source:
# #   Silver poi_content -> poi_detail.poi_content_json
# #   fallback '{}' only when Silver content is still absent.
# # Policy source:
# #   Silver poi_policy.policy_json -> poi_detail.poi_policy_json
# #   wrapped as {"policy": ..., "source_updated_at": ...} for legacy parity,
# #   fallback '{}' because poi_detail.poi_policy_json is NOT NULL.
# # Extra source:
# #   Silver poi_price -> poi_detail.poi_extra_json.price
# #   fallback '{}' because poi_detail.poi_extra_json is NOT NULL.
# _detail_sql = """
# CREATE OR REPLACE TEMP VIEW v2_gold_detail_src AS
# WITH content_pick AS (
#     SELECT *
#     FROM (
#         SELECT
#             c.*,
#             e.lang AS entity_lang,
#             ROW_NUMBER() OVER (
#                 PARTITION BY c.poi_id
#                 ORDER BY
#                     CASE
#                         WHEN c.locale = e.lang THEN 0
#                         WHEN c.locale = 'vi' THEN 1
#                         WHEN c.locale = 'en' THEN 2
#                         ELSE 3
#                     END,
#                     c.content_updated_at DESC NULLS LAST,
#                     c._updated_at DESC NULLS LAST
#             ) AS rn
#         FROM poi_content c
#         INNER JOIN poi_entity e
#             ON e.poi_id = c.poi_id
#            AND e._is_current = true
#         INNER JOIN v2_impacted_poi i
#             ON i.poi_id = c.poi_id
#     ) x
#     WHERE rn = 1
# ),
# media_ranked AS (
#     SELECT
#         m.poi_id,
#         m.media_id,
#         m.blob_url,
#         m.is_blessed,
#         m.display_order,
#         ROW_NUMBER() OVER (
#             PARTITION BY m.poi_id, __MEDIA_IDENTITY_SQL__
#             ORDER BY
#                 m._created_at ASC NULLS LAST,
#                 COALESCE(m.display_order, 2147483647) ASC,
#                 m.media_id ASC
#         ) AS dedup_rn
#     FROM poi_media m
#     INNER JOIN v2_impacted_poi i
#         ON i.poi_id = m.poi_id
#     WHERE m.is_active = true
# ),
# media_base AS (
#     SELECT
#         d.poi_id,
#         d.media_id,
#         d.blob_url,
#         d.is_blessed,
#         d.display_order,
#         ROW_NUMBER() OVER (
#             PARTITION BY d.poi_id
#             ORDER BY
#                 CASE WHEN d.is_blessed = true THEN 0 ELSE 1 END,
#                 d.display_order ASC NULLS LAST,
#                 d.media_id ASC
#         ) AS cover_rn
#     FROM media_ranked d
#     WHERE d.dedup_rn = 1
# ),
# media_agg AS (
#     SELECT
#         mb.poi_id,
#         to_json(
#             named_struct(
#                 'cover',
#                 first(
#                     CASE
#                         WHEN mb.cover_rn = 1 THEN named_struct(
#                             'media_id', mb.media_id,
#                             'blob_url', mb.blob_url
#                         )
#                     END,
#                     true
#                 ),
#                 'gallery',
#                 transform(
#                     sort_array(
#                         collect_list(
#                             named_struct(
#                                 'sort_order', COALESCE(mb.display_order, 2147483647),
#                                 'media_id', mb.media_id,
#                                 'blob_url', mb.blob_url
#                             )
#                         )
#                     ),
#                     x -> named_struct(
#                         'media_id', x.media_id,
#                         'blob_url', x.blob_url
#                     )
#                 ),
#                 'total_count',
#                 CAST(COUNT(*) AS INT)
#             )
#         ) AS poi_media_json
#     FROM media_base mb
#     GROUP BY mb.poi_id
# ),
# review_agg AS (
#     SELECT
#         rv.poi_id,
#         to_json(
#             collect_list(
#                 named_struct(
#                     'review_id', rv.review_id,
#                     'language', rv.language,
#                     'reviewer_name', rv.reviewer_name,
#                     'rating_score', rv.rating_score,
#                     'review_text', rv.review_text,
#                     'review_date', rv.review_date,
#                     'published_at', rv.published_at
#                 )
#             )
#         ) AS poi_review_json
#     FROM poi_review rv
#     INNER JOIN v2_impacted_poi i
#         ON i.poi_id = rv.poi_id
#     WHERE rv.is_visible = true
#     GROUP BY rv.poi_id
# )
# SELECT
#     e.poi_id,
#     e.lang,

#     COALESCE(ac.resolved_amenities_json, '{}') AS amenities_json,

#     COALESCE(
#         CASE
#             WHEN a.poi_id IS NULL THEN NULL
#             ELSE to_json(
#                 named_struct(
#                     'address_line1', a.address_line1,
#                     'address_line2', a.address_line2,
#                     'ward', a.ward,
#                     'district', a.district,
#                     'city', a.city,
#                     'state_province', a.state_province,
#                     'country_code', a.country_code,
#                     'postal_code', a.postal_code,
#                     'short_address', a.short_address,
#                     'full_address', a.full_address,
#                     'lat', a.lat,
#                     'lng', a.lng,
#                     'geohash', a.geohash,
#                     'h3_res8', a.h3_res8,
#                     'plus_code', a.plus_code
#                 )
#             )
#         END,
#         '{}'
#     ) AS poi_address_json,

#     COALESCE(
#         CASE
#             WHEN c.poi_id IS NULL THEN NULL
#             ELSE to_json(
#                 named_struct(
#                     'phone', c.phone,
#                     'secondary_phones', c.secondary_phones,
#                     'email', c.email,
#                     'website_url', c.website_url,
#                     'booking_url', c.booking_url,
#                     'menu_url', c.menu_url,
#                     'facebook_url', c.facebook_url,
#                     'instagram_url', c.instagram_url,
#                     'tiktok_url', c.tiktok_url
#                 )
#             )
#         END,
#         '{}'
#     ) AS poi_contact_json,

#     COALESCE(
#         CASE
#             WHEN cp.poi_id IS NULL THEN NULL
#             ELSE to_json(
#                 named_struct(
#                     'locale', cp.locale,
#                     'display_name', cp.display_name,
#                     'description_short', cp.description_short,
#                     'description_long', cp.description_long,
#                     'service_bio', cp.service_bio,
#                     'important_notice', cp.important_notice,
#                     'price_range_text', cp.price_range_text,
#                     'seo_title', cp.seo_title,
#                     'seo_description', cp.seo_description,
#                     'service_usps', cp.service_usps,
#                     'tags_text', cp.tags_text,
#                     'hours_weekday_text', cp.hours_weekday_text
#                 )
#             )
#         END,
#         '{}'
#     ) AS poi_content_json,

#     COALESCE(
#         ma.poi_media_json,
#         '{"cover":null,"gallery":[],"total_count":0}'
#     ) AS poi_media_json,

#     COALESCE(
#         CASE
#             WHEN oh.poi_id IS NULL THEN NULL
#             ELSE to_json(
#                 named_struct(
#                     'timezone', oh.timezone,
#                     'is_24_7', oh.is_24_7,
#                     'open_now', oh.open_now,
#                     'periods', oh.periods_json,
#                     'holiday_exceptions', oh.holiday_exceptions_json
#                 )
#             )
#         END,
#         '{}'
#     ) AS poi_opening_hours_json,

#     COALESCE(
#         CASE
#             WHEN pl.poi_id IS NULL THEN NULL
#             ELSE to_json(
#                 named_struct(
#                     'policy', pl.policy_json,
#                     'source_updated_at', pl.source_updated_at
#                 )
#             )
#         END,
#         '{}'
#     ) AS poi_policy_json,

#     COALESCE(
#         CASE
#             WHEN r.poi_id IS NULL THEN NULL
#             ELSE to_json(
#                 named_struct(
#                     'overall', r.overall,
#                     'total_reviews', r.total_reviews,
#                     'score_label', r.score_label,
#                     'breakdown', r.breakdown_json
#                 )
#             )
#         END,
#         '{}'
#     ) AS poi_rating_json,

#     COALESCE(ra.poi_review_json, '[]') AS poi_review_json,

#     COALESCE(
#         CASE
#             WHEN pr.poi_id IS NULL THEN NULL
#             ELSE to_json(
#                 named_struct(
#                     'price',
#                     named_struct(
#                         'currency', pr.currency,
#                         'price_level', pr.price_level,
#                         'price_min', pr.price_min,
#                         'price_max', pr.price_max
#                     )
#                 )
#             )
#         END,
#         '{}'
#     ) AS poi_extra_json,

#     e.source_id,
#     e.event_id,
#     e._created_at AS created_at,
#     current_timestamp() AS updated_at,
#     e.canonical_source AS ingestion_source,
#     e.poi_group_id

# FROM poi_entity e
# INNER JOIN v2_impacted_poi i
#     ON i.poi_id = e.poi_id
# LEFT JOIN poi_address a
#     ON a.poi_id = e.poi_id
# LEFT JOIN poi_contact c
#     ON c.poi_id = e.poi_id
# LEFT JOIN content_pick cp
#     ON cp.poi_id = e.poi_id
# LEFT JOIN poi_amenity_current ac
#     ON ac.poi_id = e.poi_id
# LEFT JOIN poi_price pr
#     ON pr.poi_id = e.poi_id
# LEFT JOIN poi_opening_hours oh
#     ON oh.poi_id = e.poi_id
# LEFT JOIN poi_policy pl
#     ON pl.poi_id = e.poi_id
# LEFT JOIN poi_rating r
#     ON r.poi_id = e.poi_id
# LEFT JOIN media_agg ma
#     ON ma.poi_id = e.poi_id
# LEFT JOIN review_agg ra
#     ON ra.poi_id = e.poi_id
# WHERE e._is_current = true
# """
# _detail_sql = _detail_sql.replace("__MEDIA_IDENTITY_SQL__", _MEDIA_IDENTITY_SQL)
# spark.sql(_detail_sql)

# # Fail early if the physical Gold target changes.
# _expected_poi_detail_cols = {
#     "poi_id","lang","amenities_json","poi_address_json","poi_contact_json",
#     "poi_content_json","poi_media_json","poi_opening_hours_json",
#     "poi_policy_json","poi_rating_json","poi_review_json","poi_extra_json",
#     "source_id","event_id","created_at","updated_at","ingestion_source",
#     "poi_group_id"
# }
# _poi_detail_target = "sgr_visitvn_dev.lh_vv_gold.dbo.poi_detail"
# _poi_detail_actual = set(spark.table(_poi_detail_target).columns)
# _poi_detail_missing = sorted(_expected_poi_detail_cols - _poi_detail_actual)

# if _poi_detail_missing:
#     raise RuntimeError(
#         "NB40_TARGET_SCHEMA_MISMATCH poi_detail missing columns: "
#         + str(_poi_detail_missing)
#     )

# print("NB40 poi_detail target schema preflight = OK")

# spark.sql("""
# MERGE INTO sgr_visitvn_dev.lh_vv_gold.dbo.poi_detail tgt
# USING v2_gold_detail_src src
# ON tgt.poi_id = src.poi_id

# WHEN MATCHED THEN UPDATE SET
#     tgt.lang                   = src.lang,
#     tgt.amenities_json         = src.amenities_json,
#     tgt.poi_address_json       = src.poi_address_json,
#     tgt.poi_contact_json       = src.poi_contact_json,
#     tgt.poi_content_json       = src.poi_content_json,
#     tgt.poi_media_json         = src.poi_media_json,
#     tgt.poi_opening_hours_json = src.poi_opening_hours_json,
#     tgt.poi_policy_json        = src.poi_policy_json,
#     tgt.poi_rating_json        = src.poi_rating_json,
#     tgt.poi_review_json        = src.poi_review_json,
#     tgt.poi_extra_json         = src.poi_extra_json,
#     tgt.source_id              = src.source_id,
#     tgt.event_id               = src.event_id,
#     tgt.ingestion_source       = src.ingestion_source,
#     tgt.poi_group_id           = src.poi_group_id,
#     tgt.updated_at             = src.updated_at

# WHEN NOT MATCHED THEN INSERT (
#     poi_id,
#     lang,
#     amenities_json,
#     poi_address_json,
#     poi_contact_json,
#     poi_content_json,
#     poi_media_json,
#     poi_opening_hours_json,
#     poi_policy_json,
#     poi_rating_json,
#     poi_review_json,
#     poi_extra_json,
#     source_id,
#     event_id,
#     created_at,
#     updated_at,
#     ingestion_source,
#     poi_group_id
# )
# VALUES (
#     src.poi_id,
#     src.lang,
#     src.amenities_json,
#     src.poi_address_json,
#     src.poi_contact_json,
#     src.poi_content_json,
#     src.poi_media_json,
#     src.poi_opening_hours_json,
#     src.poi_policy_json,
#     src.poi_rating_json,
#     src.poi_review_json,
#     src.poi_extra_json,
#     src.source_id,
#     src.event_id,
#     src.created_at,
#     src.updated_at,
#     src.ingestion_source,
#     src.poi_group_id
# )
# """)

# # -----------------------------------------------------------------------------
# # 2B. HISTORICAL JSON NOT-NULL REPAIR
# #
# # poi_detail is consumed by PostgreSQL targets where JSON payload columns are
# # NOT NULL. Incremental MERGE only touches impacted POIs, therefore historical
# # rows created by older notebook versions may still contain NULL.
# #
# # Contract:
# #   object payloads  -> '{}'
# #   poi_review_json  -> '[]'
# #   poi_media_json   -> {"cover":null,"gallery":[],"total_count":0}
# # -----------------------------------------------------------------------------
# spark.sql("""
# UPDATE sgr_visitvn_dev.lh_vv_gold.dbo.poi_detail
# SET
#     amenities_json          = COALESCE(amenities_json, '{}'),
#     poi_address_json        = COALESCE(poi_address_json, '{}'),
#     poi_contact_json        = COALESCE(poi_contact_json, '{}'),
#     poi_content_json        = COALESCE(poi_content_json, '{}'),
#     poi_media_json          = COALESCE(
#         poi_media_json,
#         '{"cover":null,"gallery":[],"total_count":0}'
#     ),
#     poi_opening_hours_json  = COALESCE(poi_opening_hours_json, '{}'),
#     poi_policy_json         = COALESCE(poi_policy_json, '{}'),
#     poi_rating_json         = COALESCE(poi_rating_json, '{}'),
#     poi_review_json         = COALESCE(poi_review_json, '[]'),
#     poi_extra_json          = COALESCE(poi_extra_json, '{}')
# WHERE
#        amenities_json         IS NULL
#     OR poi_address_json       IS NULL
#     OR poi_contact_json       IS NULL
#     OR poi_content_json       IS NULL
#     OR poi_media_json         IS NULL
#     OR poi_opening_hours_json IS NULL
#     OR poi_policy_json        IS NULL
#     OR poi_rating_json        IS NULL
#     OR poi_review_json        IS NULL
#     OR poi_extra_json         IS NULL
# """)

# # -----------------------------------------------------------------------------
# # Destination mapping for poi_master
# # poi_detail.poi_address_json.ward           -> destination_ward_mapping.ward
# # poi_detail.poi_address_json.state_province -> destination_ward_mapping.province_name
# # destination_ids / destination_names are stored as JSON-array strings.
# # -----------------------------------------------------------------------------
# spark.sql("""
# CREATE OR REPLACE TEMP VIEW v2_destination_mapping_src AS
# WITH impacted_detail AS (
#     SELECT
#         pd.poi_id,
#         trim(get_json_object(pd.poi_address_json, '$.ward')) AS ward,
#         trim(get_json_object(pd.poi_address_json, '$.state_province')) AS state_province
#     FROM sgr_visitvn_dev.lh_vv_gold.dbo.poi_detail pd
#     INNER JOIN v2_impacted_poi i
#         ON i.poi_id = pd.poi_id
#     WHERE pd.poi_address_json IS NOT NULL
# ),
# mapping_dedup AS (
#     SELECT ward, province_name, dest_id, dest_name
#     FROM (
#         SELECT
#             trim(ward) AS ward,
#             trim(province_name) AS province_name,
#             dest_id,
#             dest_name,
#             ROW_NUMBER() OVER (
#                 PARTITION BY lower(trim(ward)), lower(trim(province_name))
#                 ORDER BY dest_id
#             ) AS rn
#         FROM sgr_visitvn_dev.lh_vv_gold.dbo.destination_ward_mapping
#         WHERE ward IS NOT NULL
#           AND province_name IS NOT NULL
#           AND dest_id IS NOT NULL
#     ) x
#     WHERE rn = 1
# )
# SELECT
#     d.poi_id,
#     m.dest_id,
#     m.dest_name
# FROM impacted_detail d
# LEFT JOIN mapping_dedup m
#     ON lower(d.ward) = lower(m.ward)
#    AND lower(d.state_province) = lower(m.province_name)
# """)

# spark.sql("""
# MERGE INTO sgr_visitvn_dev.lh_vv_gold.dbo.poi_master tgt
# USING (
#     SELECT
#         poi_id,
#         CASE WHEN dest_id IS NOT NULL THEN to_json(array(dest_id)) END AS destination_ids,
#         CASE WHEN dest_name IS NOT NULL THEN to_json(array(dest_name)) END AS destination_names
#     FROM v2_destination_mapping_src
# ) src
# ON tgt.poi_id = src.poi_id
# WHEN MATCHED THEN UPDATE SET
#     tgt.destination_ids   = src.destination_ids,
#     tgt.destination_names = src.destination_names
# """)


# # -----------------------------------------------------------------------------
# # 3. Gold poi_rating, 4. Gold poi_review
# # -----------------------------------------------------------------------------
# # Gold poi_rating physical schema:
# # poi_id, overall, total_reviews, score_label, sources_json, breakdown_json,
# # sentiment_avg, source_id, event_id, updated_at, _batch_id
# _expected_poi_rating_cols = {
#     "poi_id","overall","total_reviews","score_label","sources_json",
#     "breakdown_json","sentiment_avg","source_id","event_id","updated_at","_batch_id"
# }
# _poi_rating_target = "sgr_visitvn_dev.lh_vv_gold.dbo.poi_rating"
# _poi_rating_actual = set(spark.table(_poi_rating_target).columns)
# _poi_rating_missing = sorted(_expected_poi_rating_cols - _poi_rating_actual)
# if _poi_rating_missing:
#     raise RuntimeError(
#         "NB40_TARGET_SCHEMA_MISMATCH poi_rating missing columns: "
#         + str(_poi_rating_missing)
#     )
# print("NB40 poi_rating target schema preflight = OK")

# spark.sql(f"""
# CREATE OR REPLACE TEMP VIEW v2_gold_rating_src AS
# SELECT
#     r.poi_id,
#     r.overall,
#     r.total_reviews,
#     r.score_label,
#     r.sources_json,
#     r.breakdown_json,
#     r.sentiment_avg,
#     e.source_id,
#     e.event_id,
#     current_timestamp() AS updated_at,
#     '{run_id}' AS _batch_id
# FROM poi_rating r
# INNER JOIN v2_impacted_poi i
#     ON i.poi_id = r.poi_id
# LEFT JOIN poi_entity e
#     ON e.poi_id = r.poi_id
#    AND e._is_current = true
# """)

# spark.sql("""
# MERGE INTO sgr_visitvn_dev.lh_vv_gold.dbo.poi_rating tgt
# USING v2_gold_rating_src src
# ON tgt.poi_id = src.poi_id

# WHEN MATCHED THEN UPDATE SET
#     tgt.overall        = src.overall,
#     tgt.total_reviews  = src.total_reviews,
#     tgt.score_label    = src.score_label,
#     tgt.sources_json   = src.sources_json,
#     tgt.breakdown_json = src.breakdown_json,
#     tgt.sentiment_avg  = src.sentiment_avg,
#     tgt.source_id      = src.source_id,
#     tgt.event_id       = src.event_id,
#     tgt.updated_at     = src.updated_at,
#     tgt._batch_id      = src._batch_id

# WHEN NOT MATCHED THEN INSERT (
#     poi_id,
#     overall,
#     total_reviews,
#     score_label,
#     sources_json,
#     breakdown_json,
#     sentiment_avg,
#     source_id,
#     event_id,
#     updated_at,
#     _batch_id
# )
# VALUES (
#     src.poi_id,
#     src.overall,
#     src.total_reviews,
#     src.score_label,
#     src.sources_json,
#     src.breakdown_json,
#     src.sentiment_avg,
#     src.source_id,
#     src.event_id,
#     src.updated_at,
#     src._batch_id
# )
# """)

# # -----------------------------------------------------------------------------
# # 4. Gold poi_review
# #
# # Physical Gold schema confirmed from runtime:
# # review_id, poi_id, source, source_review_id, lang, reviewer_name, rating,
# # title, positive, negative, review_text, review_date, review_trip_type,
# # sentiment_score, ranking_text, awards_json, response_text, responded_at,
# # media_urls_json, helpful_votes, published_at, last_projected_at,
# # source_id, event_id, _batch_id
# # -----------------------------------------------------------------------------

# _expected_poi_review_cols = {
#     "review_id","poi_id","source","source_review_id","lang","reviewer_name",
#     "rating","title","positive","negative","review_text","review_date",
#     "review_trip_type","sentiment_score","ranking_text","awards_json",
#     "response_text","responded_at","media_urls_json","helpful_votes",
#     "published_at","last_projected_at","source_id","event_id","_batch_id"
# }

# _poi_review_target = "sgr_visitvn_dev.lh_vv_gold.dbo.poi_review"
# _poi_review_actual = set(spark.table(_poi_review_target).columns)
# _poi_review_missing = sorted(_expected_poi_review_cols - _poi_review_actual)

# if _poi_review_missing:
#     raise RuntimeError(
#         "NB40_TARGET_SCHEMA_MISMATCH poi_review missing columns: "
#         + str(_poi_review_missing)
#     )

# print("NB40 poi_review target schema preflight = OK")

# spark.sql(f"""
# CREATE OR REPLACE TEMP VIEW v2_gold_review_src AS
# SELECT
#     r.review_id,
#     r.poi_id,
#     r.source_name AS source,
#     r.source_review_id,
#     r.language AS lang,
#     r.reviewer_name,
#     r.rating_score AS rating,
#     r.review_title AS title,

#     CAST(NULL AS STRING) AS positive,
#     CAST(NULL AS STRING) AS negative,

#     r.review_text,
#     r.review_date,
#     r.travel_type AS review_trip_type,

#     CAST(NULL AS DOUBLE) AS sentiment_score,
#     CAST(NULL AS STRING) AS ranking_text,
#     CAST(NULL AS STRING) AS awards_json,

#     r.response_text,
#     r.responded_at,
#     to_json(r.review_media_urls) AS media_urls_json,
#     r.helpful_count AS helpful_votes,

#     COALESCE(
#         r.published_at,
#         CAST(r.review_date AS TIMESTAMP),
#         r._created_at
#     ) AS published_at,

#     current_timestamp() AS last_projected_at,
#     e.source_id,
#     e.event_id,
#     '{run_id}' AS _batch_id

# FROM poi_review r
# INNER JOIN v2_impacted_poi i
#     ON i.poi_id = r.poi_id
# LEFT JOIN poi_entity e
#     ON e.poi_id = r.poi_id
#    AND e._is_current = true
# WHERE r.is_visible = true
# """)

# spark.sql("""
# MERGE INTO sgr_visitvn_dev.lh_vv_gold.dbo.poi_review tgt
# USING v2_gold_review_src src
# ON tgt.review_id = src.review_id

# WHEN MATCHED THEN UPDATE SET
#     tgt.poi_id            = src.poi_id,
#     tgt.source            = src.source,
#     tgt.source_review_id  = src.source_review_id,
#     tgt.lang              = src.lang,
#     tgt.reviewer_name     = src.reviewer_name,
#     tgt.rating            = src.rating,
#     tgt.title             = src.title,
#     tgt.positive          = src.positive,
#     tgt.negative          = src.negative,
#     tgt.review_text       = src.review_text,
#     tgt.review_date       = src.review_date,
#     tgt.review_trip_type  = src.review_trip_type,
#     tgt.sentiment_score   = src.sentiment_score,
#     tgt.ranking_text      = src.ranking_text,
#     tgt.awards_json       = src.awards_json,
#     tgt.response_text     = src.response_text,
#     tgt.responded_at      = src.responded_at,
#     tgt.media_urls_json   = src.media_urls_json,
#     tgt.helpful_votes     = src.helpful_votes,
#     tgt.published_at      = src.published_at,
#     tgt.last_projected_at = src.last_projected_at,
#     tgt.source_id         = src.source_id,
#     tgt.event_id          = src.event_id,
#     tgt._batch_id         = src._batch_id

# WHEN NOT MATCHED THEN INSERT (
#     review_id,
#     poi_id,
#     source,
#     source_review_id,
#     lang,
#     reviewer_name,
#     rating,
#     title,
#     positive,
#     negative,
#     review_text,
#     review_date,
#     review_trip_type,
#     sentiment_score,
#     ranking_text,
#     awards_json,
#     response_text,
#     responded_at,
#     media_urls_json,
#     helpful_votes,
#     published_at,
#     last_projected_at,
#     source_id,
#     event_id,
#     _batch_id
# )
# VALUES (
#     src.review_id,
#     src.poi_id,
#     src.source,
#     src.source_review_id,
#     src.lang,
#     src.reviewer_name,
#     src.rating,
#     src.title,
#     src.positive,
#     src.negative,
#     src.review_text,
#     src.review_date,
#     src.review_trip_type,
#     src.sentiment_score,
#     src.ranking_text,
#     src.awards_json,
#     src.response_text,
#     src.responded_at,
#     src.media_urls_json,
#     src.helpful_votes,
#     src.published_at,
#     src.last_projected_at,
#     src.source_id,
#     src.event_id,
#     src._batch_id
# )
# """)

# # -----------------------------------------------------------------------------
# # 5. Gold poi_enrichment_display — BUSINESS SOURCE OF TRUTH: Confluence 7A.2
# # -----------------------------------------------------------------------------
# #
# # Contract:
# #   1. Filter is_current = true
# #   2. Deduplicate latest enriched_at per (poi_id, normalized lang)
# #   3. Inner join active/current POI by poi_id
# #   4. Normalize lang: vi/en/ko only; everything else -> en
# #   5. display_category comes ONLY from final ref_business_sector
# #      exact dictionary match; unknown -> other
# #   6. Format duration / best_time / cost into language-specific display JSON
# #   7. description direct-copy translated text
# #   8. experiences keeps JSON shape; translated title/description are expected
# #      to already be canonical in poi_enrichment; keywords remain unchanged
# #   9. enrichment_job_id / validation_status are legacy Gold columns and are written NULL
# #  10. display_id = <enrichment_id>_<target_poi_id>_<lang>_display
# #
# # IMPORTANT:
# # - Do NOT use poi_entity.business_category for display_category.
# # - ref_business_sector is the canonical source for display_category.
# # - enrichment_job_id / validation_status are no longer used by business logic.
# # -----------------------------------------------------------------------------

# _expected_enrichment_display_cols = {
#     "display_id","enrichment_id","poi_id","poi_name","lang","display_category",
#     "description","time_to_spend","best_time","cost","experiences",
#     "enrichment_job_id","validation_status","display_generated_at","is_current",
#     "ingestion_source","created_at","updated_at"
# }
# _enrichment_display_target = "sgr_visitvn_dev.lh_vv_gold.dbo.poi_enrichment_display"
# _enrichment_display_actual = set(spark.table(_enrichment_display_target).columns)
# _enrichment_display_missing = sorted(
#     _expected_enrichment_display_cols - _enrichment_display_actual
# )
# if _enrichment_display_missing:
#     raise RuntimeError(
#         "NB40_TARGET_SCHEMA_MISMATCH poi_enrichment_display missing columns: "
#         + str(_enrichment_display_missing)
#     )
# print("NB40 poi_enrichment_display target schema preflight = OK")

# # --------------------------------------------------------------
# # 5.1 Canonical source + language normalization + latest dedupe
# # --------------------------------------------------------------
# spark.sql("""
# CREATE OR REPLACE TEMP VIEW v2_7a2_enrichment_latest AS
# WITH normalized AS (
#     SELECT
#         e.*,
#         CASE
#             WHEN lower(trim(e.lang)) IN ('vi','en','ko')
#                 THEN lower(trim(e.lang))
#             ELSE 'en'
#         END AS normalized_lang,

#         CASE
#             WHEN lower(trim(e.ref_business_sector)) IN (
#                 'accommodation',
#                 'f_n_b',
#                 'attraction',
#                 'tour_n_activities',
#                 'shopping_n_retail',
#                 'transport_route_base',
#                 'transport_on_demand',
#                 'vehicle_rental',
#                 'spa_n_wellness',
#                 'event',
#                 'other'
#             )
#             THEN lower(trim(e.ref_business_sector))
#             ELSE 'other'
#         END AS display_category_norm,

#         ROW_NUMBER() OVER (
#             PARTITION BY
#                 e.poi_id,
#                 CASE
#                     WHEN lower(trim(e.lang)) IN ('vi','en','ko')
#                         THEN lower(trim(e.lang))
#                     ELSE 'en'
#                 END
#             ORDER BY
#                 e.enriched_at DESC NULLS LAST,
#                 e._updated_at DESC NULLS LAST,
#                 e.enrichment_id DESC
#         ) AS rn
#     FROM poi_enrichment e
#     INNER JOIN v2_impacted_poi i
#         ON i.poi_id = e.poi_id
#     WHERE e.is_current = true
# )
# SELECT *
# FROM normalized
# WHERE rn = 1
# """)

# # --------------------------------------------------------------
# # 5.2 Active POI + localized formatter projection
# # --------------------------------------------------------------
# spark.sql("""
# CREATE OR REPLACE TEMP VIEW v2_gold_enrichment_display_src AS
# WITH src AS (
#     SELECT
#         e.*,
#         p.canonical_source AS poi_ingestion_source
#     FROM v2_7a2_enrichment_latest e
#     INNER JOIN poi_entity p
#         ON p.poi_id = e.poi_id
#        AND p._is_current = true
#        AND p.is_active = true
# ),
# labels AS (
#     SELECT
#         s.*,

#         -- Duration label
#         CASE s.normalized_lang
#             WHEN 'vi' THEN CASE s.display_category_norm
#                 WHEN 'accommodation'        THEN 'Thời gian lưu trú lý tưởng'
#                 WHEN 'f_n_b'                THEN 'Thời gian dùng bữa'
#                 WHEN 'attraction'           THEN 'Thời gian tham quan'
#                 WHEN 'tour_n_activities'    THEN 'Thời lượng hoạt động'
#                 WHEN 'shopping_n_retail'    THEN 'Thời gian mua sắm'
#                 WHEN 'transport_route_base' THEN 'Thời gian đưa đón'
#                 WHEN 'transport_on_demand'  THEN 'Thời gian thuê phổ biến'
#                 WHEN 'vehicle_rental'       THEN 'Thời gian thuê phổ biến'
#                 WHEN 'spa_n_wellness'       THEN 'Thời lượng liệu trình'
#                 WHEN 'event'                THEN 'Thời lượng chương trình'
#                 ELSE 'Thời lượng trải nghiệm'
#             END
#             WHEN 'ko' THEN CASE s.display_category_norm
#                 WHEN 'accommodation'        THEN '권장 숙박 시간'
#                 WHEN 'f_n_b'                THEN '식사 소요 시간'
#                 WHEN 'attraction'           THEN '관람 소요 시간'
#                 WHEN 'tour_n_activities'    THEN '활동 소요 시간'
#                 WHEN 'shopping_n_retail'    THEN '쇼핑 소요 시간'
#                 WHEN 'transport_route_base' THEN '이동 소요 시간'
#                 WHEN 'transport_on_demand'  THEN '일반 대여 시간'
#                 WHEN 'vehicle_rental'       THEN '일반 대여 시간'
#                 WHEN 'spa_n_wellness'       THEN '관리 소요 시간'
#                 WHEN 'event'                THEN '행사 소요 시간'
#                 ELSE '체험 소요 시간'
#             END
#             ELSE CASE s.display_category_norm
#                 WHEN 'accommodation'        THEN 'Ideal stay duration'
#                 WHEN 'f_n_b'                THEN 'Meal duration'
#                 WHEN 'attraction'           THEN 'Visit duration'
#                 WHEN 'tour_n_activities'    THEN 'Activity duration'
#                 WHEN 'shopping_n_retail'    THEN 'Shopping duration'
#                 WHEN 'transport_route_base' THEN 'Transfer duration'
#                 WHEN 'transport_on_demand'  THEN 'Common rental duration'
#                 WHEN 'vehicle_rental'       THEN 'Common rental duration'
#                 WHEN 'spa_n_wellness'       THEN 'Treatment duration'
#                 WHEN 'event'                THEN 'Event duration'
#                 ELSE 'Experience duration'
#             END
#         END AS duration_label,

#         -- Best-time label
#         CASE s.normalized_lang
#             WHEN 'vi' THEN CASE s.display_category_norm
#                 WHEN 'accommodation'        THEN 'Mùa lưu trú đẹp nhất'
#                 WHEN 'f_n_b'                THEN 'Thời điểm nên ghé'
#                 WHEN 'attraction'           THEN 'Mùa đẹp nhất'
#                 WHEN 'tour_n_activities'    THEN 'Thời điểm phù hợp'
#                 WHEN 'shopping_n_retail'    THEN 'Thời điểm nên ghé'
#                 WHEN 'transport_route_base' THEN 'Thời điểm phù hợp'
#                 WHEN 'transport_on_demand'  THEN 'Thời điểm nên thuê'
#                 WHEN 'vehicle_rental'       THEN 'Thời điểm nên thuê'
#                 WHEN 'spa_n_wellness'       THEN 'Thời điểm phù hợp'
#                 WHEN 'event'                THEN 'Thời điểm nên xem'
#                 ELSE 'Thời điểm phù hợp'
#             END
#             WHEN 'ko' THEN CASE s.display_category_norm
#                 WHEN 'accommodation'        THEN '추천 숙박 시즌'
#                 WHEN 'f_n_b'                THEN '방문하기 좋은 시간'
#                 WHEN 'attraction'           THEN '추천 시즌'
#                 WHEN 'tour_n_activities'    THEN '적합한 시기'
#                 WHEN 'shopping_n_retail'    THEN '방문하기 좋은 시간'
#                 WHEN 'transport_route_base' THEN '적합한 시간'
#                 WHEN 'transport_on_demand'  THEN '대여하기 좋은 시기'
#                 WHEN 'vehicle_rental'       THEN '대여하기 좋은 시기'
#                 WHEN 'spa_n_wellness'       THEN '적합한 시간'
#                 WHEN 'event'                THEN '참석하기 좋은 시간'
#                 ELSE '적합한 시간'
#             END
#             ELSE CASE s.display_category_norm
#                 WHEN 'accommodation'        THEN 'Best stay season'
#                 WHEN 'f_n_b'                THEN 'Best time to visit'
#                 WHEN 'attraction'           THEN 'Best season'
#                 WHEN 'tour_n_activities'    THEN 'Suitable time'
#                 WHEN 'shopping_n_retail'    THEN 'Best time to visit'
#                 WHEN 'transport_route_base' THEN 'Suitable time'
#                 WHEN 'transport_on_demand'  THEN 'Best rental time'
#                 WHEN 'vehicle_rental'       THEN 'Best rental time'
#                 WHEN 'spa_n_wellness'       THEN 'Suitable time'
#                 WHEN 'event'                THEN 'Best time to attend'
#                 ELSE 'Best time'
#             END
#         END AS best_time_label,

#         -- Cost label
#         CASE s.normalized_lang
#             WHEN 'vi' THEN CASE s.display_category_norm
#                 WHEN 'accommodation'        THEN 'Giá phòng tham khảo'
#                 WHEN 'f_n_b'                THEN 'Mức giá tham khảo'
#                 WHEN 'attraction'           THEN 'Chi phí tham quan'
#                 WHEN 'tour_n_activities'    THEN 'Chi phí tham khảo'
#                 WHEN 'shopping_n_retail'    THEN 'Chi tiêu tham khảo'
#                 WHEN 'transport_route_base' THEN 'Chi phí đưa đón'
#                 WHEN 'transport_on_demand'  THEN 'Giá thuê tham khảo'
#                 WHEN 'vehicle_rental'       THEN 'Giá thuê tham khảo'
#                 WHEN 'spa_n_wellness'       THEN 'Giá dịch vụ tham khảo'
#                 WHEN 'event'                THEN 'Giá vé tham khảo'
#                 ELSE 'Chi phí tham khảo'
#             END
#             WHEN 'ko' THEN CASE s.display_category_norm
#                 WHEN 'accommodation'        THEN '참고 객실 요금'
#                 WHEN 'f_n_b'                THEN '참고 가격'
#                 WHEN 'attraction'           THEN '관람 비용'
#                 WHEN 'tour_n_activities'    THEN '참고 비용'
#                 WHEN 'shopping_n_retail'    THEN '참고 지출'
#                 WHEN 'transport_route_base' THEN '이동 비용'
#                 WHEN 'transport_on_demand'  THEN '참고 대여 요금'
#                 WHEN 'vehicle_rental'       THEN '참고 대여 요금'
#                 WHEN 'spa_n_wellness'       THEN '참고 서비스 요금'
#                 WHEN 'event'                THEN '참고 티켓 가격'
#                 ELSE '참고 비용'
#             END
#             ELSE CASE s.display_category_norm
#                 WHEN 'accommodation'        THEN 'Reference room rate'
#                 WHEN 'f_n_b'                THEN 'Reference price'
#                 WHEN 'attraction'           THEN 'Visit cost'
#                 WHEN 'tour_n_activities'    THEN 'Reference cost'
#                 WHEN 'shopping_n_retail'    THEN 'Reference spend'
#                 WHEN 'transport_route_base' THEN 'Transfer cost'
#                 WHEN 'transport_on_demand'  THEN 'Reference rental rate'
#                 WHEN 'vehicle_rental'       THEN 'Reference rental rate'
#                 WHEN 'spa_n_wellness'       THEN 'Reference service rate'
#                 WHEN 'event'                THEN 'Reference ticket price'
#                 ELSE 'Reference cost'
#             END
#         END AS cost_label
#     FROM src s
# ),
# formatted AS (
#     SELECT
#         l.*,

#         -- Duration formatter from 7A.2.
#         CASE
#             WHEN l.time_to_spend_minutes IS NULL OR l.time_to_spend_minutes <= 0
#             THEN CASE l.normalized_lang
#                 WHEN 'vi' THEN 'Không cố định'
#                 WHEN 'ko' THEN '유동적'
#                 ELSE 'Flexible'
#             END

#             WHEN l.display_category_norm = 'accommodation'
#             THEN CASE l.normalized_lang
#                 WHEN 'vi' THEN concat(
#                     CAST(ceil(l.time_to_spend_minutes / 1440.0) AS BIGINT),
#                     ' ngày'
#                 )
#                 WHEN 'ko' THEN concat(
#                     CAST(ceil(l.time_to_spend_minutes / 1440.0) AS BIGINT),
#                     '일'
#                 )
#                 ELSE concat(
#                     CAST(ceil(l.time_to_spend_minutes / 1440.0) AS BIGINT),
#                     CASE
#                         WHEN ceil(l.time_to_spend_minutes / 1440.0) = 1 THEN ' day'
#                         ELSE ' days'
#                     END
#                 )
#             END

#             WHEN l.display_category_norm = 'tour_n_activities'
#              AND l.time_to_spend_minutes >= 420
#             THEN CASE l.normalized_lang
#                 WHEN 'vi' THEN 'Cả ngày'
#                 WHEN 'ko' THEN '종일'
#                 ELSE 'Full day'
#             END

#             WHEN l.time_to_spend_minutes < 60
#             THEN CASE l.normalized_lang
#                 WHEN 'vi' THEN concat(l.time_to_spend_minutes, ' phút')
#                 WHEN 'ko' THEN concat(l.time_to_spend_minutes, '분')
#                 ELSE concat(
#                     l.time_to_spend_minutes,
#                     CASE WHEN l.time_to_spend_minutes = 1 THEN ' minute' ELSE ' minutes' END
#                 )
#             END

#             ELSE CASE l.normalized_lang
#                 WHEN 'vi' THEN concat(
#                     CAST(floor(l.time_to_spend_minutes / 60) AS BIGINT),
#                     ' tiếng',
#                     CASE
#                         WHEN pmod(l.time_to_spend_minutes, 60) > 0
#                         THEN concat(' ', pmod(l.time_to_spend_minutes, 60), ' phút')
#                         ELSE ''
#                     END
#                 )
#                 WHEN 'ko' THEN concat(
#                     CAST(floor(l.time_to_spend_minutes / 60) AS BIGINT),
#                     '시간',
#                     CASE
#                         WHEN pmod(l.time_to_spend_minutes, 60) > 0
#                         THEN concat(' ', pmod(l.time_to_spend_minutes, 60), '분')
#                         ELSE ''
#                     END
#                 )
#                 ELSE concat(
#                     CAST(floor(l.time_to_spend_minutes / 60) AS BIGINT),
#                     CASE
#                         WHEN floor(l.time_to_spend_minutes / 60) = 1 THEN ' hour'
#                         ELSE ' hours'
#                     END,
#                     CASE
#                         WHEN pmod(l.time_to_spend_minutes, 60) > 0
#                         THEN concat(
#                             ' ',
#                             pmod(l.time_to_spend_minutes, 60),
#                             CASE
#                                 WHEN pmod(l.time_to_spend_minutes, 60) = 1
#                                 THEN ' minute'
#                                 ELSE ' minutes'
#                             END
#                         )
#                         ELSE ''
#                     END
#                 )
#             END
#         END AS duration_text,

#         -- Best-time formatter from 7A.2:
#         -- month_range is shown only when both months are valid 1..12.
#         CASE
#             WHEN lower(trim(l.best_time_type)) = 'month_range'
#              AND l.best_time_from_month BETWEEN 1 AND 12
#              AND l.best_time_to_month BETWEEN 1 AND 12
#             THEN CASE l.normalized_lang
#                 WHEN 'vi' THEN concat(
#                     'Tháng ', l.best_time_from_month,
#                     ' - Tháng ', l.best_time_to_month
#                 )
#                 WHEN 'ko' THEN concat(
#                     l.best_time_from_month, '월 - ',
#                     l.best_time_to_month, '월'
#                 )
#                 ELSE concat(
#                     element_at(
#                         array(
#                             'January','February','March','April','May','June',
#                             'July','August','September','October','November','December'
#                         ),
#                         l.best_time_from_month
#                     ),
#                     ' - ',
#                     element_at(
#                         array(
#                             'January','February','March','April','May','June',
#                             'July','August','September','October','November','December'
#                         ),
#                         l.best_time_to_month
#                     )
#                 )
#             END
#             ELSE CASE l.normalized_lang
#                 WHEN 'vi' THEN 'Quanh năm'
#                 WHEN 'ko' THEN '연중'
#                 ELSE 'Year round'
#             END
#         END AS best_time_text,

#         -- Cost formatter: round to nearest 1,000 VND; NULL stays SQL NULL.
#         CASE
#             WHEN l.cost_vnd IS NULL THEN NULL
#             ELSE CASE l.normalized_lang
#                 WHEN 'vi' THEN concat(
#                     'Từ ',
#                     format_number(round(l.cost_vnd / 1000.0) * 1000, 0),
#                     'đ'
#                 )
#                 WHEN 'ko' THEN concat(
#                     format_number(round(l.cost_vnd / 1000.0) * 1000, 0),
#                     '동부터'
#                 )
#                 ELSE concat(
#                     'From ',
#                     format_number(round(l.cost_vnd / 1000.0) * 1000, 0),
#                     ' VND'
#                 )
#             END
#         END AS cost_text
#     FROM labels l
# )
# SELECT
#     concat(
#         f.enrichment_id,
#         '_',
#         f.poi_id,
#         '_',
#         f.normalized_lang,
#         '_display'
#     ) AS display_id,

#     f.enrichment_id,
#     f.poi_id,
#     f.poi_name,
#     f.normalized_lang AS lang,
#     f.display_category_norm AS display_category,
#     f.description,

#     to_json(
#         map(
#             f.duration_label,
#             f.duration_text
#         )
#     ) AS time_to_spend,

#     to_json(
#         map(
#             f.best_time_label,
#             f.best_time_text
#         )
#     ) AS best_time,

#     CASE
#         WHEN f.cost_text IS NULL THEN NULL
#         ELSE to_json(
#             map(
#                 f.cost_label,
#                 f.cost_text
#             )
#         )
#     END AS cost,

#     f.experiences_json AS experiences,

#     CAST(NULL AS STRING) AS enrichment_job_id,
#     CAST(NULL AS STRING) AS validation_status,

#     COALESCE(f.enriched_at, f._updated_at, current_timestamp())
#         AS display_generated_at,

#     true AS is_current,

#     COALESCE(f.poi_ingestion_source, 'ENRICHMENT') AS ingestion_source,

#     COALESCE(f._created_at, f.enriched_at, current_timestamp()) AS created_at,
#     current_timestamp() AS updated_at
# FROM formatted f
# """)

# # --------------------------------------------------------------
# # 5.3 Defensive checks
# # --------------------------------------------------------------
# _dup_display_id = spark.sql("""
# SELECT COUNT(*) AS c
# FROM (
#     SELECT display_id
#     FROM v2_gold_enrichment_display_src
#     GROUP BY display_id
#     HAVING COUNT(*) > 1
# ) x
# """).first()["c"]

# if _dup_display_id > 0:
#     raise RuntimeError(
#         f"NB40_7A2_DUPLICATE_DISPLAY_ID duplicate_count={_dup_display_id}"
#     )

# _dup_poi = spark.sql("""
# SELECT COUNT(*) AS c
# FROM (
#     SELECT poi_id
#     FROM v2_gold_enrichment_display_src
#     GROUP BY poi_id
#     HAVING COUNT(*) > 1
# ) x
# """).first()["c"]

# if _dup_poi > 0:
#     raise RuntimeError(
#         f"NB40_7A2_DUPLICATE_POI duplicate_count={_dup_poi}"
#     )

# # --------------------------------------------------------------
# # 5.4 Remove stale/history rows for impacted POIs.
# # Business requirement: poi_enrichment_display keeps exactly ONE row per poi_id,
# # and that surviving row must be is_current = true.
# # --------------------------------------------------------------
# spark.sql("""
# MERGE INTO sgr_visitvn_dev.lh_vv_gold.dbo.poi_enrichment_display tgt
# USING (
#     SELECT DISTINCT poi_id, display_id
#     FROM v2_gold_enrichment_display_src
# ) src
# ON tgt.poi_id = src.poi_id
# AND tgt.display_id <> src.display_id

# WHEN MATCHED THEN DELETE
# """)

# # --------------------------------------------------------------
# # 5.5 Idempotent display MERGE
# # --------------------------------------------------------------
# spark.sql("""
# MERGE INTO sgr_visitvn_dev.lh_vv_gold.dbo.poi_enrichment_display tgt
# USING v2_gold_enrichment_display_src src
# ON tgt.display_id = src.display_id

# WHEN MATCHED THEN UPDATE SET
#     tgt.enrichment_id        = src.enrichment_id,
#     tgt.poi_id               = src.poi_id,
#     tgt.poi_name             = src.poi_name,
#     tgt.lang                 = src.lang,
#     tgt.display_category     = src.display_category,
#     tgt.description          = src.description,
#     tgt.time_to_spend        = src.time_to_spend,
#     tgt.best_time            = src.best_time,
#     tgt.cost                 = src.cost,
#     tgt.experiences          = src.experiences,
#     tgt.enrichment_job_id    = src.enrichment_job_id,
#     tgt.validation_status    = src.validation_status,
#     tgt.display_generated_at = src.display_generated_at,
#     tgt.is_current           = src.is_current,
#     tgt.ingestion_source     = src.ingestion_source,
#     tgt.updated_at           = src.updated_at

# WHEN NOT MATCHED THEN INSERT (
#     display_id,
#     enrichment_id,
#     poi_id,
#     poi_name,
#     lang,
#     display_category,
#     description,
#     time_to_spend,
#     best_time,
#     cost,
#     experiences,
#     enrichment_job_id,
#     validation_status,
#     display_generated_at,
#     is_current,
#     ingestion_source,
#     created_at,
#     updated_at
# )
# VALUES (
#     src.display_id,
#     src.enrichment_id,
#     src.poi_id,
#     src.poi_name,
#     src.lang,
#     src.display_category,
#     src.description,
#     src.time_to_spend,
#     src.best_time,
#     src.cost,
#     src.experiences,
#     src.enrichment_job_id,
#     src.validation_status,
#     src.display_generated_at,
#     src.is_current,
#     src.ingestion_source,
#     src.created_at,
#     src.updated_at
# )
# """)

# # --------------------------------------------------------------
# # 5.6 7A.2 post-merge invariants
# # --------------------------------------------------------------
# _missing_display = spark.sql("""
# SELECT COUNT(*) AS c
# FROM v2_gold_enrichment_display_src src
# LEFT ANTI JOIN sgr_visitvn_dev.lh_vv_gold.dbo.poi_enrichment_display tgt
#     ON tgt.display_id = src.display_id
#    AND tgt.is_current = true
# """).first()["c"]

# if _missing_display > 0:
#     raise RuntimeError(
#         f"NB40_7A2_DISPLAY_INVARIANT_FAILED missing_current_display={_missing_display}"
#     )

# _bad_display_cardinality = spark.sql("""
# SELECT COUNT(*) AS c
# FROM (
#     SELECT
#         poi_id,
#         COUNT(*) AS total_rows,
#         SUM(CASE WHEN is_current = true THEN 1 ELSE 0 END) AS current_rows
#     FROM sgr_visitvn_dev.lh_vv_gold.dbo.poi_enrichment_display
#     GROUP BY poi_id
#     HAVING COUNT(*) <> 1
#         OR SUM(CASE WHEN is_current = true THEN 1 ELSE 0 END) <> 1
# ) x
# """).first()["c"]

# if _bad_display_cardinality > 0:
#     raise RuntimeError(
#         f"NB40_7A2_POI_CARDINALITY_FAILED invalid_poi_count={_bad_display_cardinality}"
#     )

# _display_count = spark.sql(
#     "SELECT COUNT(*) AS c FROM v2_gold_enrichment_display_src"
# ).first()["c"]

# print(f"NB40 7A.2 enrichment_display projected = {_display_count}")

# # -----------------------------------------------------------------------------
# # Hard invariant: a publishable impacted Silver entity with source identity must
# # be materialized in Gold poi_master before NB40 succeeds.
# # -----------------------------------------------------------------------------
# missing_gold_master = spark.sql("""
# SELECT COUNT(*) AS c
# FROM (
#     SELECT DISTINCT e.poi_id
#     FROM poi_entity e
#     INNER JOIN v2_impacted_poi i ON i.poi_id=e.poi_id
#     INNER JOIN poi_source_map sm ON sm.poi_id=e.poi_id
#     LEFT ANTI JOIN sgr_visitvn_dev.lh_vv_gold.dbo.poi_master m ON m.poi_id=e.poi_id
#     WHERE e._is_current=true
#       AND e.poi_name IS NOT NULL
# ) x
# """).first()["c"]

# master_src_count = spark.sql("SELECT COUNT(*) AS c FROM v2_gold_master_src").first()["c"]
# impacted_count = spark.sql("SELECT COUNT(*) AS c FROM v2_impacted_poi").first()["c"]

# print(f"NB40 impacted_count           = {impacted_count}")
# print(f"NB40 master_src_count         = {master_src_count}")
# print(f"NB40 missing_gold_master      = {missing_gold_master}")

# if missing_gold_master > 0:
#     raise RuntimeError(
#         f"NB40_GOLD_MASTER_INVARIANT_FAILED missing_gold_master={missing_gold_master}"
#     )

# notebookutils.notebook.exit("SUCCESS")


In [ ]:
# # NB40 V7 - stable baseline + Fabric-safe media dedup
# # - Based on the last stable MEDIA_CONTRACT_FIXED notebook, not V4/V5/V6.
# # - Preserves the established namespace/execution contract.
# # - Removes correlated media aggregation and deduplicates in CTEs.

# # Fabric Notebook: NB_40_GOLD_POI_SERVING
# # READ : Silver (+ existing Gold destination reference tables)
# # WRITE: Gold poi_master, poi_detail, poi_rating, poi_review, poi_enrichment_display

# run_id = globals().get("run_id", "DEV_RUN")
# STAGE = "_stage_poi_language"

# # Only impacted logical POIs are rebuilt.
# spark.sql(f"""
# CREATE OR REPLACE TEMP VIEW v2_impacted_poi AS
# SELECT DISTINCT poi_id,poi_group_id,lang FROM {STAGE} WHERE run_id='{run_id}'
# """)

# # Optional migration columns
# for tbl in ["sgr_visitvn_dev.lh_vv_gold.dbo.poi_master", "sgr_visitvn_dev.lh_vv_gold.dbo.poi_detail"]:
#     try:
#         spark.sql(f"ALTER TABLE {tbl} ADD COLUMNS (poi_group_id STRING)")
#     except Exception:
#         pass

# # -----------------------------------------------------------------------------
# # 7A.2 BUSINESS CONTRACT PREFLIGHT
# #
# # Business source of truth:
# # poi_enrichment -> poi_enrichment_display
# #
# # NB40 intentionally does NOT infer these canonical fields from legacy columns.
# # If they are absent, NB30 / poi_enrichment schema must be aligned first.
# # -----------------------------------------------------------------------------
# _REQUIRED_7A2_POI_ENRICHMENT_COLS = {
#     "enrichment_id",
#     "poi_id",
#     "poi_name",
#     "lang",
#     "ref_business_sector",
#     "description",
#     "time_to_spend_minutes",
#     "best_time_type",
#     "best_time_from_month",
#     "best_time_to_month",
#     "cost_vnd",
#     "experiences_json",
#     "enriched_at",
#     "is_current",
#     "_created_at",
#     "_updated_at"
# }

# _actual_enrichment_cols = set(spark.table("poi_enrichment").columns)
# _missing_7a2_cols = sorted(
#     _REQUIRED_7A2_POI_ENRICHMENT_COLS - _actual_enrichment_cols
# )

# if _missing_7a2_cols:
#     raise RuntimeError(
#         "NB40_7A2_SOURCE_SCHEMA_MISMATCH: poi_enrichment is missing canonical "
#         f"business fields required by 7A.2: {_missing_7a2_cols}. "
#         "Required canonical 7A.2 source fields are missing. "
#         "Align NB30/poi_enrichment first."
#     )

# print("NB40 7A.2 source schema preflight = OK")

# # -----------------------------------------------------------------------------
# # NB40 PHYSICAL-SCHEMA PREFLIGHT
# # Fail before any Gold MERGE if a target schema drifts.
# # -----------------------------------------------------------------------------
# _TARGET_SCHEMAS = {
#     "sgr_visitvn_dev.lh_vv_gold.dbo.poi_master": {
#         "poi_id","lang","slug","poi_name","business_sector","business_category",
#         "subcategory_tags","operating_status","is_active","is_verified",
#         "short_description","short_address","district","rating_overall","rating_count",
#         "destination_ids","nearby_pois","destination_names","created_at","updated_at",
#         "version","source_id","event_id","ref_type","ref_business_sector",
#         "ingestion_source","partner_id","is_bookable","poi_group_id"
#     },
#     "sgr_visitvn_dev.lh_vv_gold.dbo.poi_detail": {
#         "poi_id","lang","amenities_json","poi_address_json","poi_contact_json",
#         "poi_content_json","poi_media_json","poi_opening_hours_json","poi_policy_json",
#         "poi_rating_json","poi_review_json","poi_extra_json","source_id","event_id",
#         "created_at","updated_at","ingestion_source","poi_group_id"
#     },
#     "sgr_visitvn_dev.lh_vv_gold.dbo.poi_rating": {
#         "poi_id","overall","total_reviews","score_label","sources_json","breakdown_json",
#         "sentiment_avg","source_id","event_id","updated_at","_batch_id"
#     },
#     "sgr_visitvn_dev.lh_vv_gold.dbo.poi_review": {
#         "review_id","poi_id","source","source_review_id","lang","reviewer_name","rating",
#         "title","positive","negative","review_text","review_date","review_trip_type",
#         "sentiment_score","ranking_text","awards_json","response_text","responded_at",
#         "media_urls_json","helpful_votes","published_at","last_projected_at",
#         "source_id","event_id","_batch_id"
#     },
#     "sgr_visitvn_dev.lh_vv_gold.dbo.poi_enrichment_display": {
#         "display_id","enrichment_id","poi_id","poi_name","lang","display_category",
#         "description","time_to_spend","best_time","cost","experiences",
#         "enrichment_job_id","validation_status","display_generated_at","is_current",
#         "ingestion_source","created_at","updated_at"
#     }
# }

# for _tbl, _expected in _TARGET_SCHEMAS.items():
#     _actual = set(spark.table(_tbl).columns)
#     _missing = sorted(_expected - _actual)
#     _unexpected = sorted(_actual - _expected)
#     if _missing:
#         raise RuntimeError(
#             f"NB40_SCHEMA_PREFLIGHT_FAILED table={_tbl} "
#             f"missing={_missing} unexpected={_unexpected}"
#         )
#     print(
#         f"NB40 schema preflight OK: {_tbl} "
#         f"(columns={len(_actual)}, unexpected={_unexpected})"
#     )

# # 1. poi_master - impacted-only and language-specific.
# # NOTE: normalize_vn() is intentionally NOT used because it is not a built-in
# # Spark SQL routine in Fabric. Province matching below uses only built-in
# # lower/trim/regexp_replace/translate functions for Vietnamese diacritic removal.
# spark.sql("""
# CREATE OR REPLACE TEMP VIEW v2_gold_master_src AS
# WITH entity AS (
#  SELECT e.* FROM poi_entity e JOIN v2_impacted_poi i ON i.poi_id=e.poi_id
#  WHERE e._is_current=true AND e.poi_name IS NOT NULL
# ), content_pick AS (
#  SELECT * FROM (
#    SELECT c.*,e.lang,
#      ROW_NUMBER() OVER(PARTITION BY c.poi_id ORDER BY CASE WHEN c.locale=e.lang THEN 0 WHEN c.locale='vi' THEN 1 WHEN c.locale='en' THEN 2 ELSE 3 END) rn
#    FROM poi_content c JOIN entity e ON e.poi_id=c.poi_id
#  ) WHERE rn=1
# ), dest_mapping AS (
#  SELECT DISTINCT ward, lower(trim(regexp_replace(translate(lower(coalesce(province_name,'')), 'àáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ', 'aaaaaaaaaaaaaaaaaeeeeeeeeeeeiiiiiooooooooooooooooouuuuuuuuuuuyyyyyd'), '\\s+', ' '))) province_name_nodiac,dest_id,dest_name
#  FROM sgr_visitvn_dev.lh_vv_gold.dbo.destination_ward_mapping WHERE ward IS NOT NULL AND province_name IS NOT NULL
# ), projected AS (
#  SELECT e.poi_id,e.poi_group_id,e.lang,e.slug,e.poi_name,e.business_sector,e.business_category,
#    to_json(e.subcategory_tags) subcategory_tags,e.operating_status,e.is_active,e.is_verified,
#    c.description_short short_description,COALESCE(a.short_address,a.full_address) short_address,a.district,
#    r.overall rating_overall,COALESCE(r.total_reviews,0) rating_count,
#    CASE WHEN dm.dest_id IS NOT NULL THEN to_json(array(dm.dest_id)) END destination_ids,
#    CASE WHEN dm.dest_name IS NOT NULL THEN to_json(array(dm.dest_name)) END destination_names,
#    e.canonical_source ingestion_source,e.partner_id,e.source_id,e.event_id,e._created_at created_at,e._updated_at updated_at,
#    CAST(NULL AS INT) version,dm.dest_id _dest_id_for_join
#  FROM entity e
#  INNER JOIN poi_source_map sm ON sm.poi_id=e.poi_id
#  LEFT JOIN poi_address a ON a.poi_id=e.poi_id
#  LEFT JOIN poi_rating r ON r.poi_id=e.poi_id
#  LEFT JOIN content_pick c ON c.poi_id=e.poi_id
#  LEFT JOIN dest_mapping dm ON dm.ward=a.ward AND dm.province_name_nodiac=lower(trim(regexp_replace(translate(lower(coalesce(a.state_province,'')), 'àáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ', 'aaaaaaaaaaaaaaaaaeeeeeeeeeeeiiiiiooooooooooooooooouuuuuuuuuuuyyyyyd'), '\\s+', ' ')))
# ), nearby AS (
#  SELECT p.poi_id,to_json(collect_list(a.poi_id)) nearby_pois
#  FROM projected p JOIN anchor_poi_map a ON a.destination_id=p._dest_id_for_join AND a.poi_id<>p.poi_id
#  GROUP BY p.poi_id
# )
# SELECT p.*,n.nearby_pois FROM projected p LEFT JOIN nearby n ON n.poi_id=p.poi_id
# """)

# spark.sql("""
# MERGE INTO sgr_visitvn_dev.lh_vv_gold.dbo.poi_master tgt
# USING v2_gold_master_src src ON tgt.poi_id=src.poi_id
# WHEN MATCHED THEN UPDATE SET
#  tgt.poi_group_id=src.poi_group_id,tgt.lang=src.lang,tgt.slug=src.slug,tgt.poi_name=src.poi_name,
#  tgt.business_sector=src.business_sector,tgt.business_category=src.business_category,tgt.subcategory_tags=src.subcategory_tags,
#  tgt.operating_status=src.operating_status,tgt.is_active=src.is_active,tgt.is_verified=src.is_verified,
#  tgt.short_description=src.short_description,tgt.short_address=src.short_address,tgt.district=src.district,
#  tgt.rating_overall=src.rating_overall,tgt.rating_count=src.rating_count,
#  tgt.destination_ids=COALESCE(src.destination_ids,tgt.destination_ids),tgt.destination_names=COALESCE(src.destination_names,tgt.destination_names),
#  tgt.nearby_pois=COALESCE(src.nearby_pois,tgt.nearby_pois),tgt.ingestion_source=src.ingestion_source,
#  tgt.partner_id=src.partner_id,tgt.source_id=src.source_id,tgt.event_id=src.event_id,tgt.updated_at=src.updated_at
# WHEN NOT MATCHED THEN INSERT(
#  poi_id,poi_group_id,lang,slug,poi_name,business_sector,business_category,subcategory_tags,operating_status,is_active,is_verified,
#  short_description,short_address,district,rating_overall,rating_count,destination_ids,destination_names,nearby_pois,ingestion_source,
#  partner_id,source_id,event_id,created_at,updated_at,version,is_bookable
# ) VALUES(
#  src.poi_id,src.poi_group_id,src.lang,src.slug,src.poi_name,src.business_sector,src.business_category,src.subcategory_tags,
#  src.operating_status,src.is_active,src.is_verified,src.short_description,src.short_address,src.district,src.rating_overall,src.rating_count,
#  src.destination_ids,src.destination_names,src.nearby_pois,src.ingestion_source,src.partner_id,src.source_id,src.event_id,
#  src.created_at,src.updated_at,COALESCE(src.version,0),false
# )
# """)

# # -----------------------------------------------------------------------------
# # MEDIA DEDUP CONTRACT - downstream consumer
# #
# # NB20 is the ONLY authoritative writer of media business identity:
# #   media_dedup_key = SHA256(source_name|source_id|TRIM(photo_api_uri))
# #
# # NB40 must not recompute the key from photo_api_uri/original_url. It only
# # consumes the persisted key and performs a defensive POI-scoped dedup before
# # building poi_detail.poi_media_json.
# #
# # Historical rows that predate media_dedup_key fall back to media_id only.
# # This fallback preserves historical rows without redefining original_url as a
# # business identity.
# # -----------------------------------------------------------------------------
# _media_cols = set(spark.table("poi_media").columns)
# if "media_dedup_key" not in _media_cols:
#     raise RuntimeError(
#         "NB40_MEDIA_SCHEMA_MISMATCH: poi_media.media_dedup_key is required. "
#         "Run the updated NB20 before NB40."
#     )

# _MEDIA_IDENTITY_SQL = (
#     "COALESCE(NULLIF(TRIM(m.media_dedup_key), ''), CONCAT('LEGACY_ID:', m.media_id))"
# )

# print(
#     "NB40 media contract = poi_id + persisted media_dedup_key "
#     "(NB20 authoritative; media_id fallback only for historical rows)"
# )

# # 2. poi_detail - packed API document, impacted only.
# # Physical Gold schema:
# #   poi_id, lang, amenities_json, poi_address_json, poi_contact_json,
# #   poi_content_json, poi_media_json, poi_opening_hours_json, poi_policy_json,
# #   poi_rating_json, poi_review_json, poi_extra_json, source_id, event_id,
# #   created_at, updated_at, ingestion_source, poi_group_id
# #
# # Optional Silver components are LEFT JOINed. They must never remove a POI.
# # Gold poi_detail JSON contract:
# #   object JSON columns -> '{}' when source data is absent
# #   poi_review_json     -> '[]' when source data is absent
# #   poi_media_json      -> {"cover":null,"gallery":[],"total_count":0}
# #
# # poi_media_json business contract:
# # {
# #   "cover":  {"media_id": "...", "blob_url": "..."} | null,
# #   "gallery": [{"media_id": "...", "blob_url": "..."}],
# #   "total_count": N
# # }
# # Cover priority: is_blessed=true -> display_order ASC -> media_id ASC.
# # Gallery contains all active media, including the selected cover.
# # This prevents downstream PostgreSQL NOT NULL violations.
# # Amenities parity with the legacy pipeline:
# #   poi_amenity_current.resolved_amenities_json -> poi_detail.amenities_json
# #   fallback '{}' when the amenity row/payload is missing.
# # Content source:
# #   Silver poi_content -> poi_detail.poi_content_json
# #   fallback '{}' only when Silver content is still absent.
# # Policy source:
# #   Silver poi_policy.policy_json -> poi_detail.poi_policy_json
# #   wrapped as {"policy": ..., "source_updated_at": ...} for legacy parity,
# #   fallback '{}' because poi_detail.poi_policy_json is NOT NULL.
# # Extra source:
# #   Silver poi_price -> poi_detail.poi_extra_json.price
# #   fallback '{}' because poi_detail.poi_extra_json is NOT NULL.
# _detail_sql = """
# CREATE OR REPLACE TEMP VIEW v2_gold_detail_src AS
# WITH content_pick AS (
#     SELECT *
#     FROM (
#         SELECT
#             c.*,
#             e.lang AS entity_lang,
#             ROW_NUMBER() OVER (
#                 PARTITION BY c.poi_id
#                 ORDER BY
#                     CASE
#                         WHEN c.locale = e.lang THEN 0
#                         WHEN c.locale = 'vi' THEN 1
#                         WHEN c.locale = 'en' THEN 2
#                         ELSE 3
#                     END,
#                     c.content_updated_at DESC NULLS LAST,
#                     c._updated_at DESC NULLS LAST
#             ) AS rn
#         FROM poi_content c
#         INNER JOIN poi_entity e
#             ON e.poi_id = c.poi_id
#            AND e._is_current = true
#         INNER JOIN v2_impacted_poi i
#             ON i.poi_id = c.poi_id
#     ) x
#     WHERE rn = 1
# ),
# media_ranked AS (
#     SELECT
#         m.poi_id,
#         m.media_id,
#         m.blob_url,
#         m.is_blessed,
#         m.display_order,
#         ROW_NUMBER() OVER (
#             PARTITION BY m.poi_id, __MEDIA_IDENTITY_SQL__
#             ORDER BY
#                 m._created_at ASC NULLS LAST,
#                 COALESCE(m.display_order, 2147483647) ASC,
#                 m.media_id ASC
#         ) AS dedup_rn
#     FROM poi_media m
#     INNER JOIN v2_impacted_poi i
#         ON i.poi_id = m.poi_id
#     WHERE m.is_active = true
# ),
# media_base AS (
#     SELECT
#         d.poi_id,
#         d.media_id,
#         d.blob_url,
#         d.is_blessed,
#         d.display_order,
#         ROW_NUMBER() OVER (
#             PARTITION BY d.poi_id
#             ORDER BY
#                 CASE WHEN d.is_blessed = true THEN 0 ELSE 1 END,
#                 d.display_order ASC NULLS LAST,
#                 d.media_id ASC
#         ) AS cover_rn
#     FROM media_ranked d
#     WHERE d.dedup_rn = 1
# ),
# media_agg AS (
#     SELECT
#         mb.poi_id,
#         to_json(
#             named_struct(
#                 'cover',
#                 first(
#                     CASE
#                         WHEN mb.cover_rn = 1 THEN named_struct(
#                             'media_id', mb.media_id,
#                             'blob_url', mb.blob_url
#                         )
#                     END,
#                     true
#                 ),
#                 'gallery',
#                 transform(
#                     sort_array(
#                         collect_list(
#                             named_struct(
#                                 'sort_order', COALESCE(mb.display_order, 2147483647),
#                                 'media_id', mb.media_id,
#                                 'blob_url', mb.blob_url
#                             )
#                         )
#                     ),
#                     x -> named_struct(
#                         'media_id', x.media_id,
#                         'blob_url', x.blob_url
#                     )
#                 ),
#                 'total_count',
#                 CAST(COUNT(*) AS INT)
#             )
#         ) AS poi_media_json
#     FROM media_base mb
#     GROUP BY mb.poi_id
# ),
# review_agg AS (
#     SELECT
#         rv.poi_id,
#         to_json(
#             collect_list(
#                 named_struct(
#                     'review_id', rv.review_id,
#                     'language', rv.language,
#                     'reviewer_name', rv.reviewer_name,
#                     'rating_score', rv.rating_score,
#                     'review_text', rv.review_text,
#                     'review_date', rv.review_date,
#                     'published_at', rv.published_at
#                 )
#             )
#         ) AS poi_review_json
#     FROM poi_review rv
#     INNER JOIN v2_impacted_poi i
#         ON i.poi_id = rv.poi_id
#     WHERE rv.is_visible = true
#     GROUP BY rv.poi_id
# )
# SELECT
#     e.poi_id,
#     e.lang,

#     COALESCE(ac.resolved_amenities_json, '{}') AS amenities_json,

#     COALESCE(
#         CASE
#             WHEN a.poi_id IS NULL THEN NULL
#             ELSE to_json(
#                 named_struct(
#                     'address_line1', a.address_line1,
#                     'address_line2', a.address_line2,
#                     'ward', a.ward,
#                     'district', a.district,
#                     'city', a.city,
#                     'state_province', a.state_province,
#                     'country_code', a.country_code,
#                     'postal_code', a.postal_code,
#                     'short_address', a.short_address,
#                     'full_address', a.full_address,
#                     'lat', a.lat,
#                     'lng', a.lng,
#                     'geohash', a.geohash,
#                     'h3_res8', a.h3_res8,
#                     'plus_code', a.plus_code
#                 )
#             )
#         END,
#         '{}'
#     ) AS poi_address_json,

#     COALESCE(
#         CASE
#             WHEN c.poi_id IS NULL THEN NULL
#             ELSE to_json(
#                 named_struct(
#                     'phone', c.phone,
#                     'secondary_phones', c.secondary_phones,
#                     'email', c.email,
#                     'website_url', c.website_url,
#                     'booking_url', c.booking_url,
#                     'menu_url', c.menu_url,
#                     'facebook_url', c.facebook_url,
#                     'instagram_url', c.instagram_url,
#                     'tiktok_url', c.tiktok_url
#                 )
#             )
#         END,
#         '{}'
#     ) AS poi_contact_json,

#     COALESCE(
#         CASE
#             WHEN cp.poi_id IS NULL THEN NULL
#             ELSE to_json(
#                 named_struct(
#                     'locale', cp.locale,
#                     'display_name', cp.display_name,
#                     'description_short', cp.description_short,
#                     'description_long', cp.description_long,
#                     'service_bio', cp.service_bio,
#                     'important_notice', cp.important_notice,
#                     'price_range_text', cp.price_range_text,
#                     'seo_title', cp.seo_title,
#                     'seo_description', cp.seo_description,
#                     'service_usps', cp.service_usps,
#                     'tags_text', cp.tags_text,
#                     'hours_weekday_text', cp.hours_weekday_text
#                 )
#             )
#         END,
#         '{}'
#     ) AS poi_content_json,

#     COALESCE(
#         ma.poi_media_json,
#         '{"cover":null,"gallery":[],"total_count":0}'
#     ) AS poi_media_json,

#     COALESCE(
#         CASE
#             WHEN oh.poi_id IS NULL THEN NULL
#             ELSE to_json(
#                 named_struct(
#                     'timezone', oh.timezone,
#                     'is_24_7', oh.is_24_7,
#                     'open_now', oh.open_now,
#                     'periods', oh.periods_json,
#                     'holiday_exceptions', oh.holiday_exceptions_json
#                 )
#             )
#         END,
#         '{}'
#     ) AS poi_opening_hours_json,

#     COALESCE(
#         CASE
#             WHEN pl.poi_id IS NULL THEN NULL
#             ELSE to_json(
#                 named_struct(
#                     'policy', pl.policy_json,
#                     'source_updated_at', pl.source_updated_at
#                 )
#             )
#         END,
#         '{}'
#     ) AS poi_policy_json,

#     COALESCE(
#         CASE
#             WHEN r.poi_id IS NULL THEN NULL
#             ELSE to_json(
#                 named_struct(
#                     'overall', r.overall,
#                     'total_reviews', r.total_reviews,
#                     'score_label', r.score_label,
#                     'breakdown', r.breakdown_json
#                 )
#             )
#         END,
#         '{}'
#     ) AS poi_rating_json,

#     COALESCE(ra.poi_review_json, '[]') AS poi_review_json,

#     COALESCE(
#         CASE
#             WHEN pr.poi_id IS NULL THEN NULL
#             ELSE to_json(
#                 named_struct(
#                     'price',
#                     named_struct(
#                         'currency', pr.currency,
#                         'price_level', pr.price_level,
#                         'price_min', pr.price_min,
#                         'price_max', pr.price_max
#                     )
#                 )
#             )
#         END,
#         '{}'
#     ) AS poi_extra_json,

#     e.source_id,
#     e.event_id,
#     e._created_at AS created_at,
#     current_timestamp() AS updated_at,
#     e.canonical_source AS ingestion_source,
#     e.poi_group_id

# FROM poi_entity e
# INNER JOIN v2_impacted_poi i
#     ON i.poi_id = e.poi_id
# LEFT JOIN poi_address a
#     ON a.poi_id = e.poi_id
# LEFT JOIN poi_contact c
#     ON c.poi_id = e.poi_id
# LEFT JOIN content_pick cp
#     ON cp.poi_id = e.poi_id
# LEFT JOIN poi_amenity_current ac
#     ON ac.poi_id = e.poi_id
# LEFT JOIN poi_price pr
#     ON pr.poi_id = e.poi_id
# LEFT JOIN poi_opening_hours oh
#     ON oh.poi_id = e.poi_id
# LEFT JOIN poi_policy pl
#     ON pl.poi_id = e.poi_id
# LEFT JOIN poi_rating r
#     ON r.poi_id = e.poi_id
# LEFT JOIN media_agg ma
#     ON ma.poi_id = e.poi_id
# LEFT JOIN review_agg ra
#     ON ra.poi_id = e.poi_id
# WHERE e._is_current = true
# """
# _detail_sql = _detail_sql.replace("__MEDIA_IDENTITY_SQL__", _MEDIA_IDENTITY_SQL)
# spark.sql(_detail_sql)

# # Fail early if the physical Gold target changes.
# _expected_poi_detail_cols = {
#     "poi_id","lang","amenities_json","poi_address_json","poi_contact_json",
#     "poi_content_json","poi_media_json","poi_opening_hours_json",
#     "poi_policy_json","poi_rating_json","poi_review_json","poi_extra_json",
#     "source_id","event_id","created_at","updated_at","ingestion_source",
#     "poi_group_id"
# }
# _poi_detail_target = "sgr_visitvn_dev.lh_vv_gold.dbo.poi_detail"
# _poi_detail_actual = set(spark.table(_poi_detail_target).columns)
# _poi_detail_missing = sorted(_expected_poi_detail_cols - _poi_detail_actual)

# if _poi_detail_missing:
#     raise RuntimeError(
#         "NB40_TARGET_SCHEMA_MISMATCH poi_detail missing columns: "
#         + str(_poi_detail_missing)
#     )

# print("NB40 poi_detail target schema preflight = OK")

# spark.sql("""
# MERGE INTO sgr_visitvn_dev.lh_vv_gold.dbo.poi_detail tgt
# USING v2_gold_detail_src src
# ON tgt.poi_id = src.poi_id

# WHEN MATCHED THEN UPDATE SET
#     tgt.lang                   = src.lang,
#     tgt.amenities_json         = src.amenities_json,
#     tgt.poi_address_json       = src.poi_address_json,
#     tgt.poi_contact_json       = src.poi_contact_json,
#     tgt.poi_content_json       = src.poi_content_json,
#     tgt.poi_media_json         = src.poi_media_json,
#     tgt.poi_opening_hours_json = src.poi_opening_hours_json,
#     tgt.poi_policy_json        = src.poi_policy_json,
#     tgt.poi_rating_json        = src.poi_rating_json,
#     tgt.poi_review_json        = src.poi_review_json,
#     tgt.poi_extra_json         = src.poi_extra_json,
#     tgt.source_id              = src.source_id,
#     tgt.event_id               = src.event_id,
#     tgt.ingestion_source       = src.ingestion_source,
#     tgt.poi_group_id           = src.poi_group_id,
#     tgt.updated_at             = src.updated_at

# WHEN NOT MATCHED THEN INSERT (
#     poi_id,
#     lang,
#     amenities_json,
#     poi_address_json,
#     poi_contact_json,
#     poi_content_json,
#     poi_media_json,
#     poi_opening_hours_json,
#     poi_policy_json,
#     poi_rating_json,
#     poi_review_json,
#     poi_extra_json,
#     source_id,
#     event_id,
#     created_at,
#     updated_at,
#     ingestion_source,
#     poi_group_id
# )
# VALUES (
#     src.poi_id,
#     src.lang,
#     src.amenities_json,
#     src.poi_address_json,
#     src.poi_contact_json,
#     src.poi_content_json,
#     src.poi_media_json,
#     src.poi_opening_hours_json,
#     src.poi_policy_json,
#     src.poi_rating_json,
#     src.poi_review_json,
#     src.poi_extra_json,
#     src.source_id,
#     src.event_id,
#     src.created_at,
#     src.updated_at,
#     src.ingestion_source,
#     src.poi_group_id
# )
# """)

# # -----------------------------------------------------------------------------
# # 2B. HISTORICAL JSON NOT-NULL REPAIR
# #
# # poi_detail is consumed by PostgreSQL targets where JSON payload columns are
# # NOT NULL. Incremental MERGE only touches impacted POIs, therefore historical
# # rows created by older notebook versions may still contain NULL.
# #
# # Contract:
# #   object payloads  -> '{}'
# #   poi_review_json  -> '[]'
# #   poi_media_json   -> {"cover":null,"gallery":[],"total_count":0}
# # -----------------------------------------------------------------------------
# spark.sql("""
# UPDATE sgr_visitvn_dev.lh_vv_gold.dbo.poi_detail
# SET
#     amenities_json          = COALESCE(amenities_json, '{}'),
#     poi_address_json        = COALESCE(poi_address_json, '{}'),
#     poi_contact_json        = COALESCE(poi_contact_json, '{}'),
#     poi_content_json        = COALESCE(poi_content_json, '{}'),
#     poi_media_json          = COALESCE(
#         poi_media_json,
#         '{"cover":null,"gallery":[],"total_count":0}'
#     ),
#     poi_opening_hours_json  = COALESCE(poi_opening_hours_json, '{}'),
#     poi_policy_json         = COALESCE(poi_policy_json, '{}'),
#     poi_rating_json         = COALESCE(poi_rating_json, '{}'),
#     poi_review_json         = COALESCE(poi_review_json, '[]'),
#     poi_extra_json          = COALESCE(poi_extra_json, '{}')
# WHERE
#        amenities_json         IS NULL
#     OR poi_address_json       IS NULL
#     OR poi_contact_json       IS NULL
#     OR poi_content_json       IS NULL
#     OR poi_media_json         IS NULL
#     OR poi_opening_hours_json IS NULL
#     OR poi_policy_json        IS NULL
#     OR poi_rating_json        IS NULL
#     OR poi_review_json        IS NULL
#     OR poi_extra_json         IS NULL
# """)

# # -----------------------------------------------------------------------------
# # Destination mapping for poi_master
# # poi_detail.poi_address_json.ward           -> destination_ward_mapping.ward
# # poi_detail.poi_address_json.state_province -> destination_ward_mapping.province_name
# # destination_ids / destination_names are stored as JSON-array strings.
# # -----------------------------------------------------------------------------
# spark.sql("""
# CREATE OR REPLACE TEMP VIEW v2_destination_mapping_src AS
# WITH impacted_detail AS (
#     SELECT
#         pd.poi_id,
#         trim(get_json_object(pd.poi_address_json, '$.ward')) AS ward,
#         trim(get_json_object(pd.poi_address_json, '$.state_province')) AS state_province
#     FROM sgr_visitvn_dev.lh_vv_gold.dbo.poi_detail pd
#     INNER JOIN v2_impacted_poi i
#         ON i.poi_id = pd.poi_id
#     WHERE pd.poi_address_json IS NOT NULL
# ),
# mapping_dedup AS (
#     SELECT ward, province_name, dest_id, dest_name
#     FROM (
#         SELECT
#             trim(ward) AS ward,
#             trim(province_name) AS province_name,
#             dest_id,
#             dest_name,
#             ROW_NUMBER() OVER (
#                 PARTITION BY lower(trim(ward)), lower(trim(province_name))
#                 ORDER BY dest_id
#             ) AS rn
#         FROM sgr_visitvn_dev.lh_vv_gold.dbo.destination_ward_mapping
#         WHERE ward IS NOT NULL
#           AND province_name IS NOT NULL
#           AND dest_id IS NOT NULL
#     ) x
#     WHERE rn = 1
# )
# SELECT
#     d.poi_id,
#     m.dest_id,
#     m.dest_name
# FROM impacted_detail d
# LEFT JOIN mapping_dedup m
#     ON lower(d.ward) = lower(m.ward)
#    AND lower(d.state_province) = lower(m.province_name)
# """)

# spark.sql("""
# MERGE INTO sgr_visitvn_dev.lh_vv_gold.dbo.poi_master tgt
# USING (
#     SELECT
#         poi_id,
#         CASE WHEN dest_id IS NOT NULL THEN to_json(array(dest_id)) END AS destination_ids,
#         CASE WHEN dest_name IS NOT NULL THEN to_json(array(dest_name)) END AS destination_names
#     FROM v2_destination_mapping_src
# ) src
# ON tgt.poi_id = src.poi_id
# WHEN MATCHED THEN UPDATE SET
#     tgt.destination_ids   = src.destination_ids,
#     tgt.destination_names = src.destination_names
# """)


# # -----------------------------------------------------------------------------
# # 3. Gold poi_rating, 4. Gold poi_review
# # -----------------------------------------------------------------------------
# # Gold poi_rating physical schema:
# # poi_id, overall, total_reviews, score_label, sources_json, breakdown_json,
# # sentiment_avg, source_id, event_id, updated_at, _batch_id
# _expected_poi_rating_cols = {
#     "poi_id","overall","total_reviews","score_label","sources_json",
#     "breakdown_json","sentiment_avg","source_id","event_id","updated_at","_batch_id"
# }
# _poi_rating_target = "sgr_visitvn_dev.lh_vv_gold.dbo.poi_rating"
# _poi_rating_actual = set(spark.table(_poi_rating_target).columns)
# _poi_rating_missing = sorted(_expected_poi_rating_cols - _poi_rating_actual)
# if _poi_rating_missing:
#     raise RuntimeError(
#         "NB40_TARGET_SCHEMA_MISMATCH poi_rating missing columns: "
#         + str(_poi_rating_missing)
#     )
# print("NB40 poi_rating target schema preflight = OK")

# spark.sql(f"""
# CREATE OR REPLACE TEMP VIEW v2_gold_rating_src AS
# SELECT
#     r.poi_id,
#     r.overall,
#     r.total_reviews,
#     r.score_label,
#     r.sources_json,
#     r.breakdown_json,
#     r.sentiment_avg,
#     e.source_id,
#     e.event_id,
#     current_timestamp() AS updated_at,
#     '{run_id}' AS _batch_id
# FROM poi_rating r
# INNER JOIN v2_impacted_poi i
#     ON i.poi_id = r.poi_id
# LEFT JOIN poi_entity e
#     ON e.poi_id = r.poi_id
#    AND e._is_current = true
# """)

# spark.sql("""
# MERGE INTO sgr_visitvn_dev.lh_vv_gold.dbo.poi_rating tgt
# USING v2_gold_rating_src src
# ON tgt.poi_id = src.poi_id

# WHEN MATCHED THEN UPDATE SET
#     tgt.overall        = src.overall,
#     tgt.total_reviews  = src.total_reviews,
#     tgt.score_label    = src.score_label,
#     tgt.sources_json   = src.sources_json,
#     tgt.breakdown_json = src.breakdown_json,
#     tgt.sentiment_avg  = src.sentiment_avg,
#     tgt.source_id      = src.source_id,
#     tgt.event_id       = src.event_id,
#     tgt.updated_at     = src.updated_at,
#     tgt._batch_id      = src._batch_id

# WHEN NOT MATCHED THEN INSERT (
#     poi_id,
#     overall,
#     total_reviews,
#     score_label,
#     sources_json,
#     breakdown_json,
#     sentiment_avg,
#     source_id,
#     event_id,
#     updated_at,
#     _batch_id
# )
# VALUES (
#     src.poi_id,
#     src.overall,
#     src.total_reviews,
#     src.score_label,
#     src.sources_json,
#     src.breakdown_json,
#     src.sentiment_avg,
#     src.source_id,
#     src.event_id,
#     src.updated_at,
#     src._batch_id
# )
# """)

# # -----------------------------------------------------------------------------
# # 4. Gold poi_review
# #
# # Physical Gold schema confirmed from runtime:
# # review_id, poi_id, source, source_review_id, lang, reviewer_name, rating,
# # title, positive, negative, review_text, review_date, review_trip_type,
# # sentiment_score, ranking_text, awards_json, response_text, responded_at,
# # media_urls_json, helpful_votes, published_at, last_projected_at,
# # source_id, event_id, _batch_id
# # -----------------------------------------------------------------------------

# _expected_poi_review_cols = {
#     "review_id","poi_id","source","source_review_id","lang","reviewer_name",
#     "rating","title","positive","negative","review_text","review_date",
#     "review_trip_type","sentiment_score","ranking_text","awards_json",
#     "response_text","responded_at","media_urls_json","helpful_votes",
#     "published_at","last_projected_at","source_id","event_id","_batch_id"
# }

# _poi_review_target = "sgr_visitvn_dev.lh_vv_gold.dbo.poi_review"
# _poi_review_actual = set(spark.table(_poi_review_target).columns)
# _poi_review_missing = sorted(_expected_poi_review_cols - _poi_review_actual)

# if _poi_review_missing:
#     raise RuntimeError(
#         "NB40_TARGET_SCHEMA_MISMATCH poi_review missing columns: "
#         + str(_poi_review_missing)
#     )

# print("NB40 poi_review target schema preflight = OK")

# spark.sql(f"""
# CREATE OR REPLACE TEMP VIEW v2_gold_review_src AS
# SELECT
#     r.review_id,
#     r.poi_id,
#     r.source_name AS source,
#     r.source_review_id,
#     r.language AS lang,
#     r.reviewer_name,
#     r.rating_score AS rating,
#     r.review_title AS title,

#     CAST(NULL AS STRING) AS positive,
#     CAST(NULL AS STRING) AS negative,

#     r.review_text,
#     r.review_date,
#     r.travel_type AS review_trip_type,

#     CAST(NULL AS DOUBLE) AS sentiment_score,
#     CAST(NULL AS STRING) AS ranking_text,
#     CAST(NULL AS STRING) AS awards_json,

#     r.response_text,
#     r.responded_at,
#     to_json(r.review_media_urls) AS media_urls_json,
#     r.helpful_count AS helpful_votes,

#     COALESCE(
#         r.published_at,
#         CAST(r.review_date AS TIMESTAMP),
#         r._created_at
#     ) AS published_at,

#     current_timestamp() AS last_projected_at,
#     e.source_id,
#     e.event_id,
#     '{run_id}' AS _batch_id

# FROM poi_review r
# INNER JOIN v2_impacted_poi i
#     ON i.poi_id = r.poi_id
# LEFT JOIN poi_entity e
#     ON e.poi_id = r.poi_id
#    AND e._is_current = true
# WHERE r.is_visible = true
# """)

# spark.sql("""
# MERGE INTO sgr_visitvn_dev.lh_vv_gold.dbo.poi_review tgt
# USING v2_gold_review_src src
# ON tgt.review_id = src.review_id

# WHEN MATCHED THEN UPDATE SET
#     tgt.poi_id            = src.poi_id,
#     tgt.source            = src.source,
#     tgt.source_review_id  = src.source_review_id,
#     tgt.lang              = src.lang,
#     tgt.reviewer_name     = src.reviewer_name,
#     tgt.rating            = src.rating,
#     tgt.title             = src.title,
#     tgt.positive          = src.positive,
#     tgt.negative          = src.negative,
#     tgt.review_text       = src.review_text,
#     tgt.review_date       = src.review_date,
#     tgt.review_trip_type  = src.review_trip_type,
#     tgt.sentiment_score   = src.sentiment_score,
#     tgt.ranking_text      = src.ranking_text,
#     tgt.awards_json       = src.awards_json,
#     tgt.response_text     = src.response_text,
#     tgt.responded_at      = src.responded_at,
#     tgt.media_urls_json   = src.media_urls_json,
#     tgt.helpful_votes     = src.helpful_votes,
#     tgt.published_at      = src.published_at,
#     tgt.last_projected_at = src.last_projected_at,
#     tgt.source_id         = src.source_id,
#     tgt.event_id          = src.event_id,
#     tgt._batch_id         = src._batch_id

# WHEN NOT MATCHED THEN INSERT (
#     review_id,
#     poi_id,
#     source,
#     source_review_id,
#     lang,
#     reviewer_name,
#     rating,
#     title,
#     positive,
#     negative,
#     review_text,
#     review_date,
#     review_trip_type,
#     sentiment_score,
#     ranking_text,
#     awards_json,
#     response_text,
#     responded_at,
#     media_urls_json,
#     helpful_votes,
#     published_at,
#     last_projected_at,
#     source_id,
#     event_id,
#     _batch_id
# )
# VALUES (
#     src.review_id,
#     src.poi_id,
#     src.source,
#     src.source_review_id,
#     src.lang,
#     src.reviewer_name,
#     src.rating,
#     src.title,
#     src.positive,
#     src.negative,
#     src.review_text,
#     src.review_date,
#     src.review_trip_type,
#     src.sentiment_score,
#     src.ranking_text,
#     src.awards_json,
#     src.response_text,
#     src.responded_at,
#     src.media_urls_json,
#     src.helpful_votes,
#     src.published_at,
#     src.last_projected_at,
#     src.source_id,
#     src.event_id,
#     src._batch_id
# )
# """)

# # -----------------------------------------------------------------------------
# # 5. Gold poi_enrichment_display — BUSINESS SOURCE OF TRUTH: Confluence 7A.2
# # -----------------------------------------------------------------------------
# #
# # Contract:
# #   1. Filter is_current = true
# #   2. Deduplicate latest enriched_at per (poi_id, normalized lang)
# #   3. Inner join active/current POI by poi_id
# #   4. Normalize lang: vi/en/ko only; everything else -> en
# #   5. display_category comes ONLY from final ref_business_sector
# #      exact dictionary match; unknown -> other
# #   6. Format duration / best_time / cost into language-specific display JSON
# #   7. description direct-copy translated text
# #   8. experiences keeps JSON shape; translated title/description are expected
# #      to already be canonical in poi_enrichment; keywords remain unchanged
# #   9. enrichment_job_id / validation_status are legacy Gold columns and are written NULL
# #  10. display_id = <enrichment_id>_<target_poi_id>_<lang>_display
# #
# # IMPORTANT:
# # - Do NOT use poi_entity.business_category for display_category.
# # - ref_business_sector is the canonical source for display_category.
# # - enrichment_job_id / validation_status are no longer used by business logic.
# # -----------------------------------------------------------------------------

# _expected_enrichment_display_cols = {
#     "display_id","enrichment_id","poi_id","poi_name","lang","display_category",
#     "description","time_to_spend","best_time","cost","experiences",
#     "enrichment_job_id","validation_status","display_generated_at","is_current",
#     "ingestion_source","created_at","updated_at"
# }
# _enrichment_display_target = "sgr_visitvn_dev.lh_vv_gold.dbo.poi_enrichment_display"
# _enrichment_display_actual = set(spark.table(_enrichment_display_target).columns)
# _enrichment_display_missing = sorted(
#     _expected_enrichment_display_cols - _enrichment_display_actual
# )
# if _enrichment_display_missing:
#     raise RuntimeError(
#         "NB40_TARGET_SCHEMA_MISMATCH poi_enrichment_display missing columns: "
#         + str(_enrichment_display_missing)
#     )
# print("NB40 poi_enrichment_display target schema preflight = OK")

# # --------------------------------------------------------------
# # 5.1 Canonical source + language normalization + latest dedupe
# # --------------------------------------------------------------
# spark.sql("""
# CREATE OR REPLACE TEMP VIEW v2_7a2_enrichment_latest AS
# WITH normalized AS (
#     SELECT
#         e.*,
#         CASE
#             WHEN lower(trim(e.lang)) IN ('vi','en','ko')
#                 THEN lower(trim(e.lang))
#             ELSE 'en'
#         END AS normalized_lang,

#         CASE
#             WHEN lower(trim(e.ref_business_sector)) IN (
#                 'accommodation',
#                 'f_n_b',
#                 'attraction',
#                 'tour_n_activities',
#                 'shopping_n_retail',
#                 'transport_route_base',
#                 'transport_on_demand',
#                 'vehicle_rental',
#                 'spa_n_wellness',
#                 'event',
#                 'other'
#             )
#             THEN lower(trim(e.ref_business_sector))
#             ELSE 'other'
#         END AS display_category_norm,

#         ROW_NUMBER() OVER (
#             PARTITION BY
#                 e.poi_id,
#                 CASE
#                     WHEN lower(trim(e.lang)) IN ('vi','en','ko')
#                         THEN lower(trim(e.lang))
#                     ELSE 'en'
#                 END
#             ORDER BY
#                 e.enriched_at DESC NULLS LAST,
#                 e._updated_at DESC NULLS LAST,
#                 e.enrichment_id DESC
#         ) AS rn
#     FROM poi_enrichment e
#     INNER JOIN v2_impacted_poi i
#         ON i.poi_id = e.poi_id
#     WHERE e.is_current = true
# )
# SELECT *
# FROM normalized
# WHERE rn = 1
# """)

# # --------------------------------------------------------------
# # 5.2 Active POI + localized formatter projection
# # --------------------------------------------------------------
# spark.sql("""
# CREATE OR REPLACE TEMP VIEW v2_gold_enrichment_display_src AS
# WITH src AS (
#     SELECT
#         e.*,
#         p.canonical_source AS poi_ingestion_source
#     FROM v2_7a2_enrichment_latest e
#     INNER JOIN poi_entity p
#         ON p.poi_id = e.poi_id
#        AND p._is_current = true
#        AND p.is_active = true
# ),
# labels AS (
#     SELECT
#         s.*,

#         -- Duration label
#         CASE s.normalized_lang
#             WHEN 'vi' THEN CASE s.display_category_norm
#                 WHEN 'accommodation'        THEN 'Thời gian lưu trú lý tưởng'
#                 WHEN 'f_n_b'                THEN 'Thời gian dùng bữa'
#                 WHEN 'attraction'           THEN 'Thời gian tham quan'
#                 WHEN 'tour_n_activities'    THEN 'Thời lượng hoạt động'
#                 WHEN 'shopping_n_retail'    THEN 'Thời gian mua sắm'
#                 WHEN 'transport_route_base' THEN 'Thời gian đưa đón'
#                 WHEN 'transport_on_demand'  THEN 'Thời gian thuê phổ biến'
#                 WHEN 'vehicle_rental'       THEN 'Thời gian thuê phổ biến'
#                 WHEN 'spa_n_wellness'       THEN 'Thời lượng liệu trình'
#                 WHEN 'event'                THEN 'Thời lượng chương trình'
#                 ELSE 'Thời lượng trải nghiệm'
#             END
#             WHEN 'ko' THEN CASE s.display_category_norm
#                 WHEN 'accommodation'        THEN '권장 숙박 시간'
#                 WHEN 'f_n_b'                THEN '식사 소요 시간'
#                 WHEN 'attraction'           THEN '관람 소요 시간'
#                 WHEN 'tour_n_activities'    THEN '활동 소요 시간'
#                 WHEN 'shopping_n_retail'    THEN '쇼핑 소요 시간'
#                 WHEN 'transport_route_base' THEN '이동 소요 시간'
#                 WHEN 'transport_on_demand'  THEN '일반 대여 시간'
#                 WHEN 'vehicle_rental'       THEN '일반 대여 시간'
#                 WHEN 'spa_n_wellness'       THEN '관리 소요 시간'
#                 WHEN 'event'                THEN '행사 소요 시간'
#                 ELSE '체험 소요 시간'
#             END
#             ELSE CASE s.display_category_norm
#                 WHEN 'accommodation'        THEN 'Ideal stay duration'
#                 WHEN 'f_n_b'                THEN 'Meal duration'
#                 WHEN 'attraction'           THEN 'Visit duration'
#                 WHEN 'tour_n_activities'    THEN 'Activity duration'
#                 WHEN 'shopping_n_retail'    THEN 'Shopping duration'
#                 WHEN 'transport_route_base' THEN 'Transfer duration'
#                 WHEN 'transport_on_demand'  THEN 'Common rental duration'
#                 WHEN 'vehicle_rental'       THEN 'Common rental duration'
#                 WHEN 'spa_n_wellness'       THEN 'Treatment duration'
#                 WHEN 'event'                THEN 'Event duration'
#                 ELSE 'Experience duration'
#             END
#         END AS duration_label,

#         -- Best-time label
#         CASE s.normalized_lang
#             WHEN 'vi' THEN CASE s.display_category_norm
#                 WHEN 'accommodation'        THEN 'Mùa lưu trú đẹp nhất'
#                 WHEN 'f_n_b'                THEN 'Thời điểm nên ghé'
#                 WHEN 'attraction'           THEN 'Mùa đẹp nhất'
#                 WHEN 'tour_n_activities'    THEN 'Thời điểm phù hợp'
#                 WHEN 'shopping_n_retail'    THEN 'Thời điểm nên ghé'
#                 WHEN 'transport_route_base' THEN 'Thời điểm phù hợp'
#                 WHEN 'transport_on_demand'  THEN 'Thời điểm nên thuê'
#                 WHEN 'vehicle_rental'       THEN 'Thời điểm nên thuê'
#                 WHEN 'spa_n_wellness'       THEN 'Thời điểm phù hợp'
#                 WHEN 'event'                THEN 'Thời điểm nên xem'
#                 ELSE 'Thời điểm phù hợp'
#             END
#             WHEN 'ko' THEN CASE s.display_category_norm
#                 WHEN 'accommodation'        THEN '추천 숙박 시즌'
#                 WHEN 'f_n_b'                THEN '방문하기 좋은 시간'
#                 WHEN 'attraction'           THEN '추천 시즌'
#                 WHEN 'tour_n_activities'    THEN '적합한 시기'
#                 WHEN 'shopping_n_retail'    THEN '방문하기 좋은 시간'
#                 WHEN 'transport_route_base' THEN '적합한 시간'
#                 WHEN 'transport_on_demand'  THEN '대여하기 좋은 시기'
#                 WHEN 'vehicle_rental'       THEN '대여하기 좋은 시기'
#                 WHEN 'spa_n_wellness'       THEN '적합한 시간'
#                 WHEN 'event'                THEN '참석하기 좋은 시간'
#                 ELSE '적합한 시간'
#             END
#             ELSE CASE s.display_category_norm
#                 WHEN 'accommodation'        THEN 'Best stay season'
#                 WHEN 'f_n_b'                THEN 'Best time to visit'
#                 WHEN 'attraction'           THEN 'Best season'
#                 WHEN 'tour_n_activities'    THEN 'Suitable time'
#                 WHEN 'shopping_n_retail'    THEN 'Best time to visit'
#                 WHEN 'transport_route_base' THEN 'Suitable time'
#                 WHEN 'transport_on_demand'  THEN 'Best rental time'
#                 WHEN 'vehicle_rental'       THEN 'Best rental time'
#                 WHEN 'spa_n_wellness'       THEN 'Suitable time'
#                 WHEN 'event'                THEN 'Best time to attend'
#                 ELSE 'Best time'
#             END
#         END AS best_time_label,

#         -- Cost label
#         CASE s.normalized_lang
#             WHEN 'vi' THEN CASE s.display_category_norm
#                 WHEN 'accommodation'        THEN 'Giá phòng tham khảo'
#                 WHEN 'f_n_b'                THEN 'Mức giá tham khảo'
#                 WHEN 'attraction'           THEN 'Chi phí tham quan'
#                 WHEN 'tour_n_activities'    THEN 'Chi phí tham khảo'
#                 WHEN 'shopping_n_retail'    THEN 'Chi tiêu tham khảo'
#                 WHEN 'transport_route_base' THEN 'Chi phí đưa đón'
#                 WHEN 'transport_on_demand'  THEN 'Giá thuê tham khảo'
#                 WHEN 'vehicle_rental'       THEN 'Giá thuê tham khảo'
#                 WHEN 'spa_n_wellness'       THEN 'Giá dịch vụ tham khảo'
#                 WHEN 'event'                THEN 'Giá vé tham khảo'
#                 ELSE 'Chi phí tham khảo'
#             END
#             WHEN 'ko' THEN CASE s.display_category_norm
#                 WHEN 'accommodation'        THEN '참고 객실 요금'
#                 WHEN 'f_n_b'                THEN '참고 가격'
#                 WHEN 'attraction'           THEN '관람 비용'
#                 WHEN 'tour_n_activities'    THEN '참고 비용'
#                 WHEN 'shopping_n_retail'    THEN '참고 지출'
#                 WHEN 'transport_route_base' THEN '이동 비용'
#                 WHEN 'transport_on_demand'  THEN '참고 대여 요금'
#                 WHEN 'vehicle_rental'       THEN '참고 대여 요금'
#                 WHEN 'spa_n_wellness'       THEN '참고 서비스 요금'
#                 WHEN 'event'                THEN '참고 티켓 가격'
#                 ELSE '참고 비용'
#             END
#             ELSE CASE s.display_category_norm
#                 WHEN 'accommodation'        THEN 'Reference room rate'
#                 WHEN 'f_n_b'                THEN 'Reference price'
#                 WHEN 'attraction'           THEN 'Visit cost'
#                 WHEN 'tour_n_activities'    THEN 'Reference cost'
#                 WHEN 'shopping_n_retail'    THEN 'Reference spend'
#                 WHEN 'transport_route_base' THEN 'Transfer cost'
#                 WHEN 'transport_on_demand'  THEN 'Reference rental rate'
#                 WHEN 'vehicle_rental'       THEN 'Reference rental rate'
#                 WHEN 'spa_n_wellness'       THEN 'Reference service rate'
#                 WHEN 'event'                THEN 'Reference ticket price'
#                 ELSE 'Reference cost'
#             END
#         END AS cost_label
#     FROM src s
# ),
# formatted AS (
#     SELECT
#         l.*,

#         -- Duration formatter from 7A.2.
#         CASE
#             WHEN l.time_to_spend_minutes IS NULL OR l.time_to_spend_minutes <= 0
#             THEN CASE l.normalized_lang
#                 WHEN 'vi' THEN 'Không cố định'
#                 WHEN 'ko' THEN '유동적'
#                 ELSE 'Flexible'
#             END

#             WHEN l.display_category_norm = 'accommodation'
#             THEN CASE l.normalized_lang
#                 WHEN 'vi' THEN concat(
#                     CAST(ceil(l.time_to_spend_minutes / 1440.0) AS BIGINT),
#                     ' ngày'
#                 )
#                 WHEN 'ko' THEN concat(
#                     CAST(ceil(l.time_to_spend_minutes / 1440.0) AS BIGINT),
#                     '일'
#                 )
#                 ELSE concat(
#                     CAST(ceil(l.time_to_spend_minutes / 1440.0) AS BIGINT),
#                     CASE
#                         WHEN ceil(l.time_to_spend_minutes / 1440.0) = 1 THEN ' day'
#                         ELSE ' days'
#                     END
#                 )
#             END

#             WHEN l.display_category_norm = 'tour_n_activities'
#              AND l.time_to_spend_minutes >= 420
#             THEN CASE l.normalized_lang
#                 WHEN 'vi' THEN 'Cả ngày'
#                 WHEN 'ko' THEN '종일'
#                 ELSE 'Full day'
#             END

#             WHEN l.time_to_spend_minutes < 60
#             THEN CASE l.normalized_lang
#                 WHEN 'vi' THEN concat(l.time_to_spend_minutes, ' phút')
#                 WHEN 'ko' THEN concat(l.time_to_spend_minutes, '분')
#                 ELSE concat(
#                     l.time_to_spend_minutes,
#                     CASE WHEN l.time_to_spend_minutes = 1 THEN ' minute' ELSE ' minutes' END
#                 )
#             END

#             ELSE CASE l.normalized_lang
#                 WHEN 'vi' THEN concat(
#                     CAST(floor(l.time_to_spend_minutes / 60) AS BIGINT),
#                     ' tiếng',
#                     CASE
#                         WHEN pmod(l.time_to_spend_minutes, 60) > 0
#                         THEN concat(' ', pmod(l.time_to_spend_minutes, 60), ' phút')
#                         ELSE ''
#                     END
#                 )
#                 WHEN 'ko' THEN concat(
#                     CAST(floor(l.time_to_spend_minutes / 60) AS BIGINT),
#                     '시간',
#                     CASE
#                         WHEN pmod(l.time_to_spend_minutes, 60) > 0
#                         THEN concat(' ', pmod(l.time_to_spend_minutes, 60), '분')
#                         ELSE ''
#                     END
#                 )
#                 ELSE concat(
#                     CAST(floor(l.time_to_spend_minutes / 60) AS BIGINT),
#                     CASE
#                         WHEN floor(l.time_to_spend_minutes / 60) = 1 THEN ' hour'
#                         ELSE ' hours'
#                     END,
#                     CASE
#                         WHEN pmod(l.time_to_spend_minutes, 60) > 0
#                         THEN concat(
#                             ' ',
#                             pmod(l.time_to_spend_minutes, 60),
#                             CASE
#                                 WHEN pmod(l.time_to_spend_minutes, 60) = 1
#                                 THEN ' minute'
#                                 ELSE ' minutes'
#                             END
#                         )
#                         ELSE ''
#                     END
#                 )
#             END
#         END AS duration_text,

#         -- Best-time formatter from 7A.2:
#         -- month_range is shown only when both months are valid 1..12.
#         CASE
#             WHEN lower(trim(l.best_time_type)) = 'month_range'
#              AND l.best_time_from_month BETWEEN 1 AND 12
#              AND l.best_time_to_month BETWEEN 1 AND 12
#             THEN CASE l.normalized_lang
#                 WHEN 'vi' THEN concat(
#                     'Tháng ', l.best_time_from_month,
#                     ' - Tháng ', l.best_time_to_month
#                 )
#                 WHEN 'ko' THEN concat(
#                     l.best_time_from_month, '월 - ',
#                     l.best_time_to_month, '월'
#                 )
#                 ELSE concat(
#                     element_at(
#                         array(
#                             'January','February','March','April','May','June',
#                             'July','August','September','October','November','December'
#                         ),
#                         l.best_time_from_month
#                     ),
#                     ' - ',
#                     element_at(
#                         array(
#                             'January','February','March','April','May','June',
#                             'July','August','September','October','November','December'
#                         ),
#                         l.best_time_to_month
#                     )
#                 )
#             END
#             ELSE CASE l.normalized_lang
#                 WHEN 'vi' THEN 'Quanh năm'
#                 WHEN 'ko' THEN '연중'
#                 ELSE 'Year round'
#             END
#         END AS best_time_text,

#         -- Cost formatter: round to nearest 1,000 VND; NULL stays SQL NULL.
#         CASE
#             WHEN l.cost_vnd IS NULL THEN NULL
#             ELSE CASE l.normalized_lang
#                 WHEN 'vi' THEN concat(
#                     'Từ ',
#                     format_number(round(l.cost_vnd / 1000.0) * 1000, 0),
#                     'đ'
#                 )
#                 WHEN 'ko' THEN concat(
#                     format_number(round(l.cost_vnd / 1000.0) * 1000, 0),
#                     '동부터'
#                 )
#                 ELSE concat(
#                     'From ',
#                     format_number(round(l.cost_vnd / 1000.0) * 1000, 0),
#                     ' VND'
#                 )
#             END
#         END AS cost_text
#     FROM labels l
# )
# SELECT
#     concat(
#         f.enrichment_id,
#         '_',
#         f.poi_id,
#         '_',
#         f.normalized_lang,
#         '_display'
#     ) AS display_id,

#     f.enrichment_id,
#     f.poi_id,
#     f.poi_name,
#     f.normalized_lang AS lang,
#     f.display_category_norm AS display_category,
#     f.description,

#     to_json(
#         map(
#             f.duration_label,
#             f.duration_text
#         )
#     ) AS time_to_spend,

#     to_json(
#         map(
#             f.best_time_label,
#             f.best_time_text
#         )
#     ) AS best_time,

#     CASE
#         WHEN f.cost_text IS NULL THEN NULL
#         ELSE to_json(
#             map(
#                 f.cost_label,
#                 f.cost_text
#             )
#         )
#     END AS cost,

#     f.experiences_json AS experiences,

#     CAST(NULL AS STRING) AS enrichment_job_id,
#     CAST(NULL AS STRING) AS validation_status,

#     COALESCE(f.enriched_at, f._updated_at, current_timestamp())
#         AS display_generated_at,

#     true AS is_current,

#     COALESCE(f.poi_ingestion_source, 'ENRICHMENT') AS ingestion_source,

#     COALESCE(f._created_at, f.enriched_at, current_timestamp()) AS created_at,
#     current_timestamp() AS updated_at
# FROM formatted f
# """)

# # --------------------------------------------------------------
# # 5.3 Defensive checks
# # --------------------------------------------------------------
# _dup_display_id = spark.sql("""
# SELECT COUNT(*) AS c
# FROM (
#     SELECT display_id
#     FROM v2_gold_enrichment_display_src
#     GROUP BY display_id
#     HAVING COUNT(*) > 1
# ) x
# """).first()["c"]

# if _dup_display_id > 0:
#     raise RuntimeError(
#         f"NB40_7A2_DUPLICATE_DISPLAY_ID duplicate_count={_dup_display_id}"
#     )

# _dup_poi = spark.sql("""
# SELECT COUNT(*) AS c
# FROM (
#     SELECT poi_id
#     FROM v2_gold_enrichment_display_src
#     GROUP BY poi_id
#     HAVING COUNT(*) > 1
# ) x
# """).first()["c"]

# if _dup_poi > 0:
#     raise RuntimeError(
#         f"NB40_7A2_DUPLICATE_POI duplicate_count={_dup_poi}"
#     )

# # --------------------------------------------------------------
# # 5.4 Remove stale/history rows for impacted POIs.
# # Business requirement: poi_enrichment_display keeps exactly ONE row per poi_id,
# # and that surviving row must be is_current = true.
# # --------------------------------------------------------------
# spark.sql("""
# MERGE INTO sgr_visitvn_dev.lh_vv_gold.dbo.poi_enrichment_display tgt
# USING (
#     SELECT DISTINCT poi_id, display_id
#     FROM v2_gold_enrichment_display_src
# ) src
# ON tgt.poi_id = src.poi_id
# AND tgt.display_id <> src.display_id

# WHEN MATCHED THEN DELETE
# """)

# # --------------------------------------------------------------
# # 5.5 Idempotent display MERGE
# # --------------------------------------------------------------
# spark.sql("""
# MERGE INTO sgr_visitvn_dev.lh_vv_gold.dbo.poi_enrichment_display tgt
# USING v2_gold_enrichment_display_src src
# ON tgt.display_id = src.display_id

# WHEN MATCHED THEN UPDATE SET
#     tgt.enrichment_id        = src.enrichment_id,
#     tgt.poi_id               = src.poi_id,
#     tgt.poi_name             = src.poi_name,
#     tgt.lang                 = src.lang,
#     tgt.display_category     = src.display_category,
#     tgt.description          = src.description,
#     tgt.time_to_spend        = src.time_to_spend,
#     tgt.best_time            = src.best_time,
#     tgt.cost                 = src.cost,
#     tgt.experiences          = src.experiences,
#     tgt.enrichment_job_id    = src.enrichment_job_id,
#     tgt.validation_status    = src.validation_status,
#     tgt.display_generated_at = src.display_generated_at,
#     tgt.is_current           = src.is_current,
#     tgt.ingestion_source     = src.ingestion_source,
#     tgt.updated_at           = src.updated_at

# WHEN NOT MATCHED THEN INSERT (
#     display_id,
#     enrichment_id,
#     poi_id,
#     poi_name,
#     lang,
#     display_category,
#     description,
#     time_to_spend,
#     best_time,
#     cost,
#     experiences,
#     enrichment_job_id,
#     validation_status,
#     display_generated_at,
#     is_current,
#     ingestion_source,
#     created_at,
#     updated_at
# )
# VALUES (
#     src.display_id,
#     src.enrichment_id,
#     src.poi_id,
#     src.poi_name,
#     src.lang,
#     src.display_category,
#     src.description,
#     src.time_to_spend,
#     src.best_time,
#     src.cost,
#     src.experiences,
#     src.enrichment_job_id,
#     src.validation_status,
#     src.display_generated_at,
#     src.is_current,
#     src.ingestion_source,
#     src.created_at,
#     src.updated_at
# )
# """)

# # --------------------------------------------------------------
# # 5.6 7A.2 post-merge invariants
# # --------------------------------------------------------------
# _missing_display = spark.sql("""
# SELECT COUNT(*) AS c
# FROM v2_gold_enrichment_display_src src
# LEFT ANTI JOIN sgr_visitvn_dev.lh_vv_gold.dbo.poi_enrichment_display tgt
#     ON tgt.display_id = src.display_id
#    AND tgt.is_current = true
# """).first()["c"]

# if _missing_display > 0:
#     raise RuntimeError(
#         f"NB40_7A2_DISPLAY_INVARIANT_FAILED missing_current_display={_missing_display}"
#     )

# _bad_display_cardinality = spark.sql("""
# SELECT COUNT(*) AS c
# FROM (
#     SELECT
#         poi_id,
#         COUNT(*) AS total_rows,
#         SUM(CASE WHEN is_current = true THEN 1 ELSE 0 END) AS current_rows
#     FROM sgr_visitvn_dev.lh_vv_gold.dbo.poi_enrichment_display
#     GROUP BY poi_id
#     HAVING COUNT(*) <> 1
#         OR SUM(CASE WHEN is_current = true THEN 1 ELSE 0 END) <> 1
# ) x
# """).first()["c"]

# if _bad_display_cardinality > 0:
#     raise RuntimeError(
#         f"NB40_7A2_POI_CARDINALITY_FAILED invalid_poi_count={_bad_display_cardinality}"
#     )

# _display_count = spark.sql(
#     "SELECT COUNT(*) AS c FROM v2_gold_enrichment_display_src"
# ).first()["c"]

# print(f"NB40 7A.2 enrichment_display projected = {_display_count}")

# # -----------------------------------------------------------------------------
# # Hard invariant: a publishable impacted Silver entity with source identity must
# # be materialized in Gold poi_master before NB40 succeeds.
# # -----------------------------------------------------------------------------
# missing_gold_master = spark.sql("""
# SELECT COUNT(*) AS c
# FROM (
#     SELECT DISTINCT e.poi_id
#     FROM poi_entity e
#     INNER JOIN v2_impacted_poi i ON i.poi_id=e.poi_id
#     INNER JOIN poi_source_map sm ON sm.poi_id=e.poi_id
#     LEFT ANTI JOIN sgr_visitvn_dev.lh_vv_gold.dbo.poi_master m ON m.poi_id=e.poi_id
#     WHERE e._is_current=true
#       AND e.poi_name IS NOT NULL
# ) x
# """).first()["c"]

# master_src_count = spark.sql("SELECT COUNT(*) AS c FROM v2_gold_master_src").first()["c"]
# impacted_count = spark.sql("SELECT COUNT(*) AS c FROM v2_impacted_poi").first()["c"]

# print(f"NB40 impacted_count           = {impacted_count}")
# print(f"NB40 master_src_count         = {master_src_count}")
# print(f"NB40 missing_gold_master      = {missing_gold_master}")

# if missing_gold_master > 0:
#     raise RuntimeError(
#         f"NB40_GOLD_MASTER_INVARIANT_FAILED missing_gold_master={missing_gold_master}"
#     )

# notebookutils.notebook.exit("SUCCESS")
